# Figure 1D–J and Supplementary Figures 1–3: CHIRAL benchmarking

This self-contained R notebook regenerates the analyses, source tables, individual panels,
and assembled publication figures. Figure 1 starts at **D**. Supplementary Figure 1 shows
mouse-tumor rotations, Supplementary Figure 2 mouse-liver CR-spread rotations, and
Supplementary Figure 3 the five-panel benchmark and sampling-window diagnostics.

**Execution:** run all cells with the R (`ir`) kernel. The default reuses existing raw
CHIRAL predictions, then recomputes benchmark scoring and summary statistics before plotting.
Set `CHRONO_REFIT=true` (or set `RUN_ANALYSIS <- TRUE` in setup) to regenerate predictions
from processed expression matrices. Full refitting includes 1,500 primary runs and the
sampling-window perturbations; it is substantially more expensive than figure regeneration.
The CHIRAL engine, preprocessing, scoring, and plotting code are embedded below; no legacy
notebook or project R script is sourced at runtime.

**Input boundary:** these benchmark cohorts use existing count/TPM matrices and collection-time
metadata in `data/benchmarking/`. These are processed inputs, not sequencing reads. Their
connection to `process.smk` output files has not been established; this notebook does not
claim that they are directly generated by that workflow. The input manifest records hashes.
Public execution requires the data files listed in the companion README.

**Interpretation:** known collection times are held out from ordinary inference and used
for benchmark scoring. `best_global` applies one truth-selected sign and circular rotation
per complete run; `raw` preserves absolute orientation. Seeds are computational replicates,
not biological replicates. H–I show qualitative concordance of genes involved in inference,
not independent validation. The reference-guided method retains the historical internal ID
`modified_pre_fix_peak_trough` to preserve the benchmark provenance.

**Publication contract:** 52.5 mm square modules; widths 1–4 modules; globally configured
text with a 7 pt minimum (Figure 1 annotations/subtitles: 6 pt by explicit request); matching sizes for equivalent text roles; minimal padding;
individual and assembled PNG/PDF exports; panel letters only in PDFs, except S1/S2,
which are whole figures without panel letters in either format. Display each S1/S2 whole
figure once inline; display all individual panels and assemblies for Figure 1 and S3. Tables go to `data/publication/fig01_chiral_benchmark/` and figures to
`data/figures/publication/fig01_chiral_benchmark/`.


## 1. Setup, paths, and shared publication style


In [ ]:
# Cell 1: Setup, paths, fonts, and export helpers
required_packages <- c("dplyr", "tidyr", "purrr", "tibble", "readr", "ggplot2", "patchwork", "ragg",
    "scales", "edgeR", "digest", "jsonlite", "foreach", "doParallel")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages)) stop("Install required R packages before running: ", paste(missing_packages,
    collapse = ", "))
suppressPackageStartupMessages(invisible(lapply(required_packages, library, character.only = TRUE)))

# Set TRUE to rerun CHIRAL from processed count/TPM matrices, including the sampling-window stress
# test. FALSE reuses the validated prediction caches.  Full refitting is computationally
# substantial (1,500 primary fits plus stress fits).
RUN_ANALYSIS <- identical(tolower(Sys.getenv("CHRONO_REFIT", "false")), "true")
RNG_SEED_BASE <- 160300L
N_SEEDS <- 30L
ITERATIONS <- 500L
N_CORES <- 1L
RNGkind(kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
set.seed(RNG_SEED_BASE)

root_candidates <- unique(c(Sys.getenv("CHRONO_PROJECT_ROOT", ""), getwd(), dirname(getwd()), file.path(Sys.getenv("WORKSPACE_ROOT",
    "/home/wdeng3/workspace/Codespace"), "PDAC_Rhythmicity")))
root_candidates <- root_candidates[nzchar(root_candidates)]
root_candidates <- root_candidates[dir.exists(file.path(root_candidates, "data", "benchmarking")) | dir.exists(file.path(root_candidates,
    "notebook_publication"))]
if (!length(root_candidates)) stop("Set CHRONO_PROJECT_ROOT to the publication repository.")
PROJECT_ROOT <- normalizePath(root_candidates[[1]], winslash = "/", mustWork = TRUE)
INPUT_DIR <- Sys.getenv("CHRONO_BENCHMARK_INPUT_DIR", file.path(PROJECT_ROOT, "data", "benchmarking"))
LEGACY_BENCHMARK_DIR <- Sys.getenv("CHRONO_BENCHMARK_CACHE_DIR", file.path(INPUT_DIR, "1.6.3_chiral"))
ANALYSIS_SLUG <- "fig01_chiral_benchmark"
TABLE_DIR <- file.path(PROJECT_ROOT, "data", "publication", ANALYSIS_SLUG)
FIGURE_DIR <- file.path(PROJECT_ROOT, "data", "figures", "publication", ANALYSIS_SLUG)
LOG_DIR <- file.path(PROJECT_ROOT, "data", "logs", "publication", ANALYSIS_SLUG)
BENCHMARK_DIR <- file.path(TABLE_DIR, "benchmark")
POINT5_DIR <- file.path(BENCHMARK_DIR, "point5_window_stress_test")
invisible(lapply(c(TABLE_DIR, FIGURE_DIR, LOG_DIR), dir.create, recursive = TRUE, showWarnings = FALSE))
invisible(lapply(c(BENCHMARK_DIR, POINT5_DIR), dir.create, recursive = TRUE, showWarnings = FALSE))
cat("Mode:", if (RUN_ANALYSIS) "Refit from processed matrices" else "Validated cached inference; regenerate figure analyses",
    "\n")

# All physical sizes and text roles are configured here, in mm and points.
PANEL_MM <- 210/4
# User-requested Figure 1 H-J height exception: 33% above one square module.
FIGURE1_BOTTOM_HEIGHT_UNITS <- 1.33
FONT_FAMILY <- "Helvetica"
FONT <- list(base = 8, title = 9, axis_title = 8, tick = 7, legend = 7, strip = 8, annotation = 7, tag = 10,
    subtitle = 7)
DEFAULT_FONT <- FONT
# Figure 1-only exception requested by the user: smaller titles and 6 pt annotations.
FIGURE1_FONT <- modifyList(DEFAULT_FONT, list(title = DEFAULT_FONT$title - 1,
    strip = DEFAULT_FONT$strip - 1, annotation = DEFAULT_FONT$annotation - 1,
    subtitle = DEFAULT_FONT$subtitle - 1))
ANNOTATION_COLOUR <- "black"
stopifnot(all(unlist(DEFAULT_FONT) >= 7), all(unlist(FIGURE1_FONT) >= 6))
publication_theme <- function() {
    theme_classic(base_size = FONT$base, base_family = FONT_FAMILY) + theme(text = element_text(size = FONT$base,
        family = FONT_FAMILY, colour = "black"), axis.title = element_text(size = FONT$axis_title), axis.text = element_text(size = FONT$tick,
        colour = "black"), axis.line = element_line(linewidth = 0.3), axis.ticks = element_line(linewidth = 0.3),
        legend.title = element_text(size = FONT$legend), legend.text = element_text(size = FONT$legend),
        strip.text = element_text(size = FONT$strip, face = "bold"), strip.background = element_rect(fill = "#F2F4F5",
            colour = NA), plot.title = element_text(size = FONT$title, face = "bold", hjust = 0), plot.subtitle = element_text(size = FONT$subtitle,
            hjust = 0), plot.caption = element_text(size = FONT$annotation), plot.tag = element_text(size = FONT$tag,
            face = "bold"), plot.margin = margin(2, 2, 2, 2, unit = "mm"), legend.key = element_blank(),
        legend.background = element_blank())
}
theme_set(publication_theme())
EXPORT_MANIFEST <- list()
figure_font_sizes <- function(plot) {
    if (grDevices::dev.cur() == 1L) {
        grDevices::pdf(file = NULL)
        on.exit(grDevices::dev.off(), add = TRUE)
    }
    grob <- if (inherits(plot, "patchwork"))
        patchwork::patchworkGrob(plot) else ggplot2::ggplotGrob(plot)
    collect <- function(x) {
        here <- if (inherits(x, "text"))
            x$gp$fontsize else numeric()
        children <- c(if (!is.null(x$grobs)) x$grobs else list(), if (!is.null(x$children)) as.list(x$children) else list())
        c(here, unlist(lapply(children, collect), use.names = FALSE))
    }
    collect(grob)
}
save_publication <- function(plot, stem, width_units, height_units, pdf_plot = NULL, pdf_has_panel_labels = TRUE, display_inline = TRUE) {
    fractional_height_allowed <- grepl("^figure1/Figure1(H_|I_|J_|_assembled_)", stem) &&
        height_units %in% c(FIGURE1_BOTTOM_HEIGHT_UNITS, 1 + FIGURE1_BOTTOM_HEIGHT_UNITS)
    stopifnot(width_units %in% 1:4, is.finite(height_units), height_units >= 1,
        height_units == as.integer(height_units) || fractional_height_allowed)
    if (is.null(pdf_plot))
        stop("Provide the PDF plot for ", stem)
    font_sizes <- c(figure_font_sizes(plot), figure_font_sizes(pdf_plot))
    if (!length(font_sizes) || any(font_sizes < min(unlist(FONT)) - 1e-06))
        stop("Text below the configured font minimum found in ", stem)
    width_mm <- PANEL_MM * width_units
    height_mm <- PANEL_MM * height_units
    out_stem <- file.path(FIGURE_DIR, stem)
    dir.create(dirname(out_stem), recursive = TRUE, showWarnings = FALSE)
    pdf_file <- paste0(out_stem, ".pdf")
    png_file <- paste0(out_stem, ".png")
    grDevices::pdf(pdf_file, width = width_mm/25.4, height = height_mm/25.4, family = FONT_FAMILY, bg = "white",
        onefile = TRUE, useDingbats = FALSE)
    tryCatch(print(pdf_plot), finally = grDevices::dev.off())
    ragg::agg_png(png_file, width = width_mm, height = height_mm, units = "mm", res = 300, background = "white")
    tryCatch(print(plot), finally = grDevices::dev.off())
    EXPORT_MANIFEST[[length(EXPORT_MANIFEST) + 1L]] <<- tibble(stem = stem, width_modules = width_units,
        height_modules = height_units, width_mm = width_mm, height_mm = height_mm, png_has_panel_labels = FALSE,
        minimum_text_pt = min(font_sizes), pdf_has_panel_labels = pdf_has_panel_labels, png = file.path(stem |>
            dirname(), basename(png_file)), pdf = file.path(stem |>
            dirname(), basename(pdf_file)))
    if (display_inline && requireNamespace("IRdisplay", quietly = TRUE)) {
        IRdisplay::display_png(file = png_file, width = min(1100L, round(width_mm/25.4 * 110)))
    } else if (display_inline) print(plot)
    invisible(c(png = png_file, pdf = pdf_file))
}



## 2. Benchmark preprocessing, inference, and scoring

The analysis preserves the methods in `1.6.3_benchmarking.ipynb`: six clock-gene symbols
(`Per1/Per2/Per3/Bmal1/Tef/Dbp`; canonical input identifier `Arntl`, uppercase in human datasets), the original reference phases,
30 paired seeds, 500 iterations, edgeR log CPM followed by row-wise z-scoring for counts,
and log2(TPM + 0.25) followed by row-wise z-scoring for TPM. Coverage is reported rather than
silently imputing missing panel genes. Expression matrices use gene symbols; no new genome
alignment or annotation build is introduced by this notebook.


## Self-contained benchmark analysis

In [ ]:
# Cell 2: Processed inputs, circular metrics, and cohort preprocessing
`%||%` <- function(x, y) if (is.null(x)) y else x
log_cpm <- function(counts, normalized.lib.sizes = FALSE, log = TRUE, prior.count = 0.25) {
    if (requireNamespace("edgeR", quietly = TRUE))
        return(edgeR::cpm(counts, normalized.lib.sizes = normalized.lib.sizes, log = log, prior.count = prior.count))
    lib <- colSums(counts, na.rm = TRUE)
    lib[!is.finite(lib) | lib <= 0] <- 1
    out <- sweep(as.matrix(counts), 2, lib/1e+06, "/")
    if (log)
        log2(out + prior.count) else out
}
row_zscore <- function(expr) {
    z <- t(scale(t(as.matrix(expr))))
    z[rowSums(!is.finite(z)) == 0, , drop = FALSE]
}
`%||%` <- function(x, y) if (is.null(x)) y else x

hours_to_rad <- function(hours) (hours%%24) * pi/12
rad_to_hours <- function(phi) (phi * 12/pi)%%24

circular_signed_error_h <- function(predicted_h, truth_h) {
    ((predicted_h - truth_h + 12)%%24) - 12
}

circular_abs_error_h <- function(predicted_h, truth_h) {
    abs(circular_signed_error_h(predicted_h, truth_h))
}

circular_distance_h <- function(a_h, b_h) {
    abs(((a_h - b_h + 12)%%24) - 12)
}

in_circular_window <- function(time_h, start_h, width_h) {
    time_h <- time_h%%24
    start_h <- start_h%%24
    if (width_h >= 24)
        return(rep(TRUE, length(time_h)))
    end_h <- (start_h + width_h)%%24
    if (end_h < start_h || isTRUE(all.equal(end_h, 0)) && start_h > 0) {
        time_h >= start_h | time_h <= end_h
    } else {
        time_h >= start_h & time_h <= end_h
    }
}

fit_best_circular_alignment <- function(predicted_h, truth_h, allow_flip = TRUE, n_offsets = 200L) {
    signs <- if (allow_flip)
        c(1, -1) else 1
    # Match original CHIRAL delta.phi(): 200 equally spaced global offsets.
    offsets <- (seq_len(n_offsets)/n_offsets * 24)%%24
    best <- tibble(sign = 1, offset_h = 0, median_abs_error_h = Inf, mean_abs_error_h = Inf)

    for (s in signs) {
        error_by_offset <- vapply(offsets, function(offset_h) {
            circular_abs_error_h((s * predicted_h + offset_h)%%24, truth_h)
        }, numeric(length(predicted_h)))
        median_by_offset <- apply(error_by_offset, 2, median, na.rm = TRUE)
        best_offset_index <- which.min(median_by_offset)
        candidate <- tibble(sign = s, offset_h = offsets[[best_offset_index]], median_abs_error_h = median_by_offset[[best_offset_index]],
            mean_abs_error_h = mean(error_by_offset[, best_offset_index], na.rm = TRUE))
        # Preserve original CHIRAL's strict median-improvement rule and first-tie behavior.
        if (candidate$median_abs_error_h < best$median_abs_error_h) {
            best <- candidate
        }
    }
    best
}

score_prediction_group <- function(df) {
    raw <- df %>%
        mutate(alignment = "raw", predicted_eval_h = predicted_h%%24, alignment_sign = 1, alignment_offset_h = 0,
            signed_error_h = circular_signed_error_h(predicted_eval_h, truth_h), abs_error_h = abs(signed_error_h))

    fit <- fit_best_circular_alignment(df$predicted_h, df$truth_h, allow_flip = TRUE)
    aligned_predicted_h <- (fit$sign * df$predicted_h + fit$offset_h)%%24

    aligned <- df %>%
        mutate(alignment = "best_global", predicted_eval_h = aligned_predicted_h, alignment_sign = fit$sign,
            alignment_offset_h = fit$offset_h, signed_error_h = circular_signed_error_h(predicted_eval_h,
                truth_h), abs_error_h = abs(signed_error_h))

    bind_rows(raw, aligned)
}

alignment_specs <- tribble(~alignment, ~alignment_label, ~truth_used_to_adjust_prediction, ~deployable,
    "raw", "Raw MedAE - absolute orientation", FALSE, TRUE, "best_global", "CHIRAL-native symmetry-adjusted MedAE",
    TRUE, FALSE)

bootstrap_mean_ci <- function(x, n_boot = 2000, seed = 1603) {
    x <- x[is.finite(x)]
    if (length(x) == 0)
        return(tibble(ci_low = NA_real_, ci_high = NA_real_))
    set.seed(seed)
    boot <- replicate(n_boot, mean(sample(x, length(x), replace = TRUE)))
    tibble(ci_low = unname(quantile(boot, 0.025)), ci_high = unname(quantile(boot, 0.975)))
}
make_sample_info <- function(sample_info, sample_col, group_col, time_col) {
    sample_info %>%
        transmute(sample_id = .data[[sample_col]], group = as.character(.data[[group_col]]), timepoint_h = as.numeric(.data[[time_col]])%%24) %>%
        distinct(sample_id, .keep_all = TRUE) %>%
        as.data.frame()
}
load_mouse_tumor <- function() {
    sample_info <- read.delim(file.path(INPUT_DIR, "Testing_Tumor_Mouse_sample_info.csv"), check.names = FALSE)
    raw_counts <- read.delim(file.path(INPUT_DIR, "Testing_Tumor_Mouse_raw_counts.csv"), row.names = 1,
        check.names = FALSE)
    expr <- row_zscore(log_cpm(raw_counts, normalized.lib.sizes = FALSE, log = TRUE, prior.count = 0.25))

    list(id = "mouse_tumor", label = "Mouse tumor", species = "mouse", reference_phase_offset_h = 0,
        reference_phase_convention = "mouse reference phases: Per1/Bmal1/Dbp = pi, 1.8*pi, 2*pi/3", expr = expr,
        sample_info = make_sample_info(sample_info, "sample_id", "cell", "timepoint"), clock_genes = c("Per1",
            "Per2", "Per3", "Arntl", "Tef", "Dbp"), ref_genes = c("Per1", "Arntl", "Dbp"), ref_phases = c(pi,
            1.8 * pi, 2 * pi/3))
}

load_mouse_liver_cr_spread <- function() {
    exp <- read.delim(file.path(INPUT_DIR, "mouse_liver", "cr_spread_6mon.txt"), check.names = FALSE)
    exp$GN <- vapply(strsplit(as.character(exp$Gene), "_"), `[`, character(1), 2)

    exp_summed <- exp %>%
        group_by(GN) %>%
        summarise(across(where(is.numeric), sum), .groups = "drop") %>%
        as.data.frame()
    rownames(exp_summed) <- exp_summed$GN
    exp_summed <- exp_summed[, setdiff(colnames(exp_summed), "GN"), drop = FALSE]

    sample_cols <- colnames(exp_summed)
    sample_info <- tibble(sample_id = sample_cols, group = paste0("Rep_", substr(sample_cols, nchar(sample_cols),
        nchar(sample_cols))), timepoint = as.numeric(substr(sample_cols, nchar(sample_cols) - 2, nchar(sample_cols) -
        1))%%24)
    expr <- row_zscore(log_cpm(exp_summed, normalized.lib.sizes = FALSE, log = TRUE, prior.count = 0.25))

    list(id = "mouse_liver_cr_spread", label = "Mouse liver CR spread", species = "mouse", reference_phase_offset_h = 0,
        reference_phase_convention = "mouse reference phases: Per1/Bmal1/Dbp = pi, 1.8*pi, 2*pi/3", expr = expr,
        sample_info = make_sample_info(sample_info, "sample_id", "group", "timepoint"), clock_genes = c("Per1",
            "Per2", "Per3", "Arntl", "Tef", "Dbp"), ref_genes = c("Per1", "Arntl", "Dbp"), ref_phases = c(pi,
            1.8 * pi, 2 * pi/3))
}

load_tumor_cell <- function() {
    sample_info <- read.delim(file.path(INPUT_DIR, "Testing_Tumor_Cell_sample_info.csv"), check.names = FALSE)
    raw_counts <- read.delim(file.path(INPUT_DIR, "Testing_Tumor_Cell_raw_counts.csv"), row.names = 1,
        check.names = FALSE)
    expr <- row_zscore(log_cpm(raw_counts, normalized.lib.sizes = FALSE, log = TRUE, prior.count = 0.25))

    list(id = "tumor_cell", label = "Tumor cell lines", species = "human", reference_phase_offset_h = 0,
        reference_phase_convention = "human reference phases: PER1/BMAL1/DBP = 0, 5*pi/6, pi/2", expr = expr,
        sample_info = make_sample_info(sample_info, "sample_id", "cell", "timepoint"), clock_genes = c("PER1",
            "PER2", "PER3", "ARNTL", "TEF", "DBP"), ref_genes = c("PER1", "ARNTL", "DBP"), ref_phases = c(0,
            5 * pi/6, pi/2))
}

load_tumor_pdo <- function() {
    sample_info <- read.csv(file.path(INPUT_DIR, "Testing_Tumor_PDO_sample_info.csv"), check.names = FALSE)
    exp <- read.delim(file.path(INPUT_DIR, "Testing_Tumor_PDO_raw_TPM.txt"), row.names = 1, check.names = FALSE)
    rownames(exp) <- toupper(rownames(exp))
    colnames(exp) <- make.unique(colnames(exp))

    sample_info <- make_sample_info(sample_info, "sample_id", "rep", "timepoint")
    sample_info <- sample_info[sample_info$sample_id %in% colnames(exp), , drop = FALSE]

    expr <- row_zscore(log2(as.matrix(exp[, sample_info$sample_id, drop = FALSE]) + 0.25))

    list(id = "tumor_pdo", label = "Tumor PDO", species = "human", reference_phase_offset_h = 0, reference_phase_convention = "human reference phases: PER1/BMAL1/DBP = 0, 5*pi/6, pi/2",
        expr = expr, sample_info = sample_info, clock_genes = c("PER1", "PER2", "PER3", "ARNTL", "TEF",
            "DBP"), ref_genes = c("PER1", "ARNTL", "DBP"), ref_phases = c(0, 5 * pi/6, pi/2))
}

load_training_wb <- function() {
    exp <- read.delim(file.path(INPUT_DIR, "Training_Normal_WB_raw_TPM.txt"), row.names = 1, check.names = FALSE)
    exp$GN <- vapply(strsplit(as.character(rownames(exp)), "__"), `[`, character(1), 1)
    exp_merged <- aggregate(. ~ GN, data = exp, FUN = sum)
    rownames(exp_merged) <- exp_merged$GN
    exp_merged <- exp_merged[, setdiff(colnames(exp_merged), "GN"), drop = FALSE]

    sample_info <- read.csv(file.path(INPUT_DIR, "Training_Normal_WB_sample_info.csv"), check.names = FALSE)
    sample_info <- make_sample_info(sample_info, "sample_id", "rep", "timepoint")
    sample_info <- sample_info[sample_info$sample_id %in% colnames(exp_merged), , drop = FALSE]

    expr <- row_zscore(log2(as.matrix(exp_merged[, sample_info$sample_id, drop = FALSE]) + 0.25))

    list(id = "normal_wb_training", label = "Normal whole blood training", species = "human", reference_phase_offset_h = 0,
        reference_phase_convention = "human reference phases: PER1/BMAL1/DBP = 0, 5*pi/6, pi/2", expr = expr,
        sample_info = sample_info, clock_genes = c("PER1", "PER2", "PER3", "ARNTL", "TEF", "DBP"), ref_genes = c("PER1",
            "ARNTL", "DBP"), ref_phases = c(0, 5 * pi/6, pi/2))
}

benchmark_datasets <- list(mouse_tumor = load_mouse_tumor(), mouse_liver_cr_spread = load_mouse_liver_cr_spread(),
    tumor_cell = load_tumor_cell(), tumor_pdo = load_tumor_pdo(), normal_wb_training = load_training_wb())

map_dfr(benchmark_datasets, function(ds) {
    tibble(dataset = ds$id, species = ds$species %||% NA_character_, ref_phase_offset_h = ds$reference_phase_offset_h %||%
        0, n_genes = nrow(ds$expr), n_samples = nrow(ds$sample_info), clock_genes_present = sum(toupper(ds$clock_genes) %in%
        toupper(rownames(ds$expr))), ref_genes_present = sum(toupper(ds$ref_genes) %in% toupper(rownames(ds$expr))))
})
window_specs <- tibble::tribble(~window, ~start_h, ~width_h, ~window_label, "full_24h", 0, 24, "Full 24h",
    "0_12h", 0, 12, "0-12h", "6_18h", 6, 12, "6-18h", "12_24h", 12, 12, "12-24h", "18_6h", 18, 12, "18-6h")
method_specs <- tibble::tribble(~method, ~engine, ~method_label, ~truth_during_inference, ~deployable_absolute_phase,
    "original_default", "legacy", "Original CHIRAL, default init", FALSE, FALSE, "modified_pre_fix_peak_trough",
    "pre_fix", "Modified CHIRAL (reference init + window)", FALSE, TRUE)
STABILITY_METHODS <- method_specs$method
STABILITY_SEEDS <- seq_len(N_SEEDS)
stopifnot(exists("LEGACY_BENCHMARK_DIR"))
LEGACY_POINT5_DIR <- file.path(LEGACY_BENCHMARK_DIR, "point5_window_stress_test")
BENCHMARK_DIR <- file.path(TABLE_DIR, "benchmark")
POINT5_DIR <- file.path(BENCHMARK_DIR, "point5_window_stress_test")
OUTPUT_DIR <- BENCHMARK_DIR
dir.create(BENCHMARK_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(POINT5_DIR, recursive = TRUE, showWarnings = FALSE)
required_inputs <- c(mouse_tumor_info = file.path(INPUT_DIR, "Testing_Tumor_Mouse_sample_info.csv"),
    mouse_tumor_expr = file.path(INPUT_DIR, "Testing_Tumor_Mouse_raw_counts.csv"), mouse_liver_expr = file.path(INPUT_DIR,
        "mouse_liver", "cr_spread_6mon.txt"), tumor_cell_info = file.path(INPUT_DIR, "Testing_Tumor_Cell_sample_info.csv"),
    tumor_cell_expr = file.path(INPUT_DIR, "Testing_Tumor_Cell_raw_counts.csv"), tumor_pdo_info = file.path(INPUT_DIR,
        "Testing_Tumor_PDO_sample_info.csv"), tumor_pdo_expr = file.path(INPUT_DIR, "Testing_Tumor_PDO_raw_TPM.txt"),
    normal_wb_info = file.path(INPUT_DIR, "Training_Normal_WB_sample_info.csv"), normal_wb_expr = file.path(INPUT_DIR,
        "Training_Normal_WB_raw_TPM.txt"))
stopifnot(all(file.exists(required_inputs)))
benchmark_input_qc <- purrr::map_dfr(benchmark_datasets, function(ds) tibble::tibble(dataset = ds$id,
    n_genes = nrow(ds$expr), n_samples = nrow(ds$sample_info), clock_genes_present = sum(toupper(ds$clock_genes) %in%
        toupper(rownames(ds$expr))), reference_genes_present = sum(toupper(ds$ref_genes) %in% toupper(rownames(ds$expr)))))
stopifnot(all(benchmark_input_qc$n_samples >= 4L))
print(benchmark_input_qc)



In [ ]:
# Cell 3: Embedded CHIRAL engine Embedded production CHIRAL definitions, isolated from plotting
# helpers.  Original and modified use this same engine with their prescribed flags.
legacy_chiral <- new.env(parent = globalenv())
evalq({
    #' CHIRAL with Dual Penalty System: Sampling Window + Reference Phase Constraints
    #'
    #' Enhanced version of CHIRAL that combines sampling window constraints with reference phase penalties
    #' to improve absolute phase inference accuracy and consistency.
    #'
    #' @param E Required matrix of gene expression. Samples should be on the columns, genes on the rows.
    #' @param iterations Number of maximum iterations, default 500.
    #' @param clockgenes Set of clock genes to be used for the inference. Default NULL.
    #' @param tau2 Tau parameter for the prior on the gene coefficient, default NULL (auto-calculated).
    #' @param u u parameter for the prior on the gene means, default NULL (auto-calculated).
    #' @param sigma2 Standard deviation of the data points, default NULL (auto-calculated).
    #' @param TSM Switches the two state model on/off, default TRUE.
    #' @param mean.centre.E Center around empirical mean of the data, default TRUE.
    #' @param pbar Shows progress bar, default TRUE.
    #' @param phi.start Initial guess for phases, default NULL.
    #' @param standardize Standardize the matrix, default FALSE.
    #' @param GTEX_names Convert GTEx names format, default FALSE.
    #' @param constrain_12h_window Enable 12h sampling window constraint, default FALSE.
    #' @param window_start Start of sampling window in radians [0, 2π), default 0.
    #' @param window_width Width of sampling window in radians, default 2π.
    #' @param window_prior_strength Strength of window constraint, default 2.0.
    #' @param post_hoc_hard_constrain Enable hard constrain post hoc to force all samples fall in designated time window, default FALSE
    #' @param reference_genes Vector of gene names to use as phase references, default NULL.
    #' @param reference_phases Expected peak phases for reference genes in radians, default NULL.
    #' @param reference_troughs Expected trough phases for reference genes in radians, default NULL (auto-calculated as peak + π).
    #' @param reference_strength Strength of reference phase constraint, default 1.0.
    #' @param use_trough_constraint Whether to use trough time constraints in addition to peak, default TRUE.
    #' @param auto_adjust_anti_phase Automatically detect and correct π-shift (anti-phase) after convergence, default TRUE.
    #' @param reference_sample_indices Indices of samples corresponding to reference genes (if applicable), default NULL.
    #' @param n_cores Number of CPU cores to use for parallel processing, default NULL (auto-detect).
    #' @return List containing inferred phases, parameters, and convergence information.
    #' @examples
    #' # Basic usage with window constraint only
    #' result1 <- CHIRAL_dual_penalty(data, constrain_12h_window=TRUE, window_start=pi/4, window_width=pi)
    #'
    #' # With both window and reference constraints (peak + trough)
    #' ref_genes <- c('PER1', 'CLOCK', 'BMAL1')
    #' ref_phases <- c(pi/2, pi, 3*pi/2)  # Peaks: 6AM, 12PM, 6PM
    #' ref_troughs <- c(3*pi/2, 0, pi/2)  # Troughs: 6PM, 12AM, 6AM (optional, auto-calculated if NULL)
    #' result2 <- CHIRAL_dual_penalty(data,
    #'                               constrain_12h_window=TRUE, window_start=0, window_width=pi,
    #'                               reference_genes=ref_genes, reference_phases=ref_phases,
    #'                               reference_troughs=ref_troughs,  # Optional
    #'                               window_prior_strength=2.0, reference_strength=1.5,
    #'                               use_trough_constraint=TRUE)
    #' @export
    if (requireNamespace("tictoc", quietly = TRUE))
        library(tictoc)
    library(foreach)
    library(doParallel)

    CHIRAL_dual_penalty <- function(E, iterations = 500, clockgenes = NULL, tau2 = NULL, u = NULL, sigma2 = NULL,
        TSM = TRUE, mean.centre.E = TRUE, q = 0.1, update.q = FALSE, pbar = TRUE, phi.start = NULL, standardize = FALSE,
        GTEx_names = FALSE, constrain_12h_window = FALSE, window_start = 0, window_width = 2 * pi, window_prior_strength = 2,
        post_hoc_hard_constrain = FALSE, reference_genes = NULL, reference_phases = NULL, reference_troughs = NULL,
        reference_strength = 1, use_trough_constraint = TRUE, auto_adjust_anti_phase = TRUE, reference_sample_indices = NULL,
        n_cores = NULL, reference_decay_enabled = TRUE, decay_midpoint = 0.05, decay_sharpness = 100) {

        # Setup parallel backend
        if (is.null(n_cores)) {
            n_cores <- parallel::detectCores() - 1  # Leave one core free
        }
        if (n_cores <= 1) {
            foreach::registerDoSEQ()
        } else {
            cl <- makeCluster(n_cores)
            registerDoParallel(cl)
            on.exit(stopCluster(cl))
        }

        # Load required functions from original CHIRAL script

        # Validate inputs
        if (!is.null(reference_genes) && !is.null(reference_phases)) {
            if (length(reference_genes) != length(reference_phases)) {
                stop("reference_genes and reference_phases must have same length")
            }

            # Auto-calculate trough phases if not provided (peak + π)
            if (is.null(reference_troughs) && use_trough_constraint) {
                reference_troughs <- (reference_phases + pi)%%(2 * pi)
                message("Auto-calculated trough phases as peak + π")
            }

            # Validate trough phases if provided
            if (!is.null(reference_troughs) && use_trough_constraint) {
                if (length(reference_troughs) != length(reference_genes)) {
                  stop("reference_troughs must have same length as reference_genes")
                }
            }
        }

        # E is the data matrix - genes on rows, samples on columns
        id = as.vector(c(1, 1, 1, 1))
        E = as.matrix(E)

        # Validate window parameters
        if (constrain_12h_window && window_width > pi) {
            warning("12h window constraint requested but window_width > π. Setting window_width = π")
            window_width = pi
        }

        # Normalize window parameters
        window_start = window_start%%(2 * pi)
        window_end = (window_start + window_width)%%(2 * pi)

        E = E[, colSums(is.na(E)) == 0]
        if (standardize == TRUE) {
            E = sweep(E, 1, apply(E, 1, sd), FUN = "/")
        }
        E.full = E

        ### Clock gene selection ###
        rownames(E) = toupper(rownames(E))
        if (is.null(clockgenes)) {
            if (is.null(rownames(E))) {
                rownames(E) = 1:nrow(E)
            }
            clockgenes = rownames(E)
        }
        clockgenes = toupper(clockgenes)
        gene.list = rownames(E)

        E.full = E

        if (GTEx_names) {
            gene.list = gsub("^.*_", "", gene.list)
            gene.list = gsub("\\|.*$", "", gene.list)
        }
        clock.coord = match(clockgenes, gene.list)
        clock.coord = clock.coord[!is.na(clock.coord)]
        E = E[clock.coord, ]
        geni = gene.list[clock.coord]

        # Prepare reference gene mapping
        reference_gene_indices <- NULL
        if (!is.null(reference_genes)) {
            reference_genes <- toupper(reference_genes)
            reference_gene_indices <- match(reference_genes, geni)
            reference_gene_indices <- reference_gene_indices[!is.na(reference_gene_indices)]

            if (length(reference_gene_indices) == 0) {
                warning("No reference genes found in the dataset. Reference penalty will be ignored.")
                reference_genes <- NULL
                reference_phases <- NULL
            } else {
                # Adjust reference_phases to match found genes
                valid_ref_mask <- !is.na(match(reference_genes, geni))
                if (sum(valid_ref_mask) < length(reference_genes)) {
                  warning(paste("Some reference genes not found:", paste(reference_genes[!valid_ref_mask],
                    collapse = ", ")))
                  reference_genes <- reference_genes[valid_ref_mask]
                  reference_phases <- reference_phases[valid_ref_mask]
                }
            }
        }

        if (is.matrix(E) == FALSE) {
            return()
        }
        if (mean.centre.E == TRUE) {
            E = sweep(E, 1, rowMeans(E), FUN = "-")
        }

        ### Initialization of parameters ###
        if (is.null(sigma2)) {
            sigma2 = mean(apply(E, 1, var))
        }
        if (is.null(u)) {
            u = 0.2
        }
        if (is.null(tau2)) {
            if (constrain_12h_window) {
                tau2 = 2/(12 + ncol(E)) * window_prior_strength
            } else {
                tau2 = 4/(24 + ncol(E))
            }
        }
        if (nrow(E) > 500) {
            tau2 = tau2/10
        }
        phi = phi.start

        ### Phase initialization with dual constraints ###
        if (is.null(phi.start)) {
            # Choose initialization strategy based on reference gene availability
            if (!is.null(reference_genes) && !is.null(reference_phases)) {
                if (constrain_12h_window) {
                  message("### Using reference-based initialization WITH window constraint ###")
                } else {
                  message("### Using reference-based initialization (no window constraint) ###")
                }

                # First, get fallback initialization using spin-glass
                beta = 1000
                J = J.tilde(E)
                Zeta = Zeta.mf.ordered(J, beta, ncol(E))
                fallback_init <- Zeta[, 2] + runif(ncol(E), -0.5, 0.5)

                # Use reference gene expression to initialize phases IMPORTANT: Pass window
                # parameters to initialization function This ensures candidates are ONLY tested
                # within the valid sampling window
                phi_init <- initialize_phases_by_reference(E, reference_gene_indices, reference_phases,
                  reference_troughs, constrain_window = constrain_12h_window, window_start = window_start,
                  window_width = window_width, fallback_init = fallback_init)

                # No need for additional window constraint - already handled in initialization But
                # apply it anyway as a safety check
                if (constrain_12h_window) {
                  phi_init <- constrain_phases_to_window(phi_init, window_start, window_width)
                }
            } else {
                message("### Using standard spin-glass initialization ###")
                # Standard initialization without reference genes
                beta = 1000
                J = J.tilde(E)
                Zeta = Zeta.mf.ordered(J, beta, ncol(E))
                phi_init <- Zeta[, 2] + runif(ncol(E), -0.5, 0.5)

                # Apply window constraint if enabled
                if (constrain_12h_window) {
                  phi_init <- constrain_phases_to_window(phi_init, window_start, window_width)
                }
            }

            phi <- phi_init%%(2 * pi)
        }

        if (pbar) {
            pb = txtProgressBar(min = 0, max = iterations, style = 3)
        }

        Ng = length(E[, 1])
        N = length(E[1, ])
        Ns = N
        sigma2.0 = sigma2
        T = diag(3) * tau2
        T[1, 1] = u^2
        dTinv = 1/det(T)
        sigma2.m0 = apply(E, 1, var)
        sigma2.m1 = sigma2.m0

        phi.0 = phi
        c = cos(phi)
        s = sin(phi)
        one = rep(1, Ns)
        X = cbind(one, c, s)
        curvature = matrix(0, nrow = 2, ncol = Ns)
        S = t(E) %*% E/Ng

        S <- list()
        for (l in 1:Ng) {
            S[[l]] = E[l, ] %o% E[l, ]
        }

        W <- rep(1, nrow(E))
        W.0 <- W

        # Initialize decay tracking
        decay_history <- numeric(iterations)
        current_decay_factor <- 1

        print("Start enhanced EM procedure")
        print(paste("Total iteration:", iterations))
        if (reference_decay_enabled && !is.null(reference_genes)) {
            print(paste("Reference penalty decay enabled: midpoint =", decay_midpoint, ", sharpness =",
                decay_sharpness))
        }
        ### Start of Enhanced EM procedure ###
        for (i in 1:iterations) {
            phiold = phi
            Xold = X
            sigma2old = sigma2
            W.old = W

            Nn = (t(X) %*% X)
            Nninv <- solve(Nn)
            Tinv = solve(T)
            M = Nn + sigma2 * Tinv
            Minv = solve(M)

            alpha = Minv %*% t(X) %*% t(E)
            alpha <- t(alpha)
            colnames(alpha) = c("mu", "a", "b")
            alphat = as.matrix(alpha)
            alpha = as.data.frame(alpha)
            if (TSM == FALSE) {
                W <- W.0
            }
            Tot <- sum(W)

            A = sum(W * (alpha$a * alpha$a/sigma2 + Minv[2, 2]))/Tot
            B = sum(W * (alpha$a * alpha$b/sigma2 + Minv[2, 3]))/Tot
            C = sum(W * (alpha$b * alpha$a/sigma2 + Minv[3, 2]))/Tot
            D = sum(W * (alpha$b * alpha$b/sigma2 + Minv[3, 3]))/Tot

            K = matrix(c(A, B, C, D), nrow = 2, ncol = 2, byrow = T)
            if (any(is.nan(K))) {
                return(list(alpha = alpha, weights = W, iteration = i))
            }

            al = apply(E, 2, function(x) {
                return(sum(W * (alpha$a * (x - alpha$mu)/sigma2 - Minv[1, 2]))/Tot)
            })
            be = apply(E, 2, function(x) {
                return(sum(W * (alpha$b * (x - alpha$mu)/sigma2 - Minv[1, 3]))/Tot)
            })
            O = as.matrix(rbind(al, be))
            if (any(is.nan(O))) {
                return(list(alpha = alpha, weights = W, iteration = i))
            }

            ### Calculate sigmoid decay factor based on convergence (delta_phi) ###
            if (i > 1) {
                delta_phi <- max(abs(phi - phiold))
            } else {
                delta_phi <- 1  # First iteration: no delta_phi yet, use full penalty
            }

            # Sigmoid decay: factor = 1 / (1 + exp(sharpness * (midpoint - delta_phi))) When
            # delta_phi >> midpoint: factor ≈ 1 (full penalty, still exploring) When delta_phi ≈
            # midpoint: factor ≈ 0.5 (transitioning) When delta_phi << midpoint: factor ≈ 0 (no
            # penalty, converged)
            if (reference_decay_enabled && !is.null(reference_genes)) {
                current_decay_factor <- 1/(1 + exp(decay_sharpness * (decay_midpoint - delta_phi)))
            } else {
                current_decay_factor <- 1  # No decay if disabled
            }

            # Store decay history
            decay_history[i] <- current_decay_factor

            ### Enhanced root finding with dual penalties ###
            rooted = apply(O, 2, function(x) {
                zero = B * B * C * C + A * A * D * D - x[1] * x[1] * (D * D + C * C) - x[2] * x[2] *
                  (A * A + B * B) + 2 * x[1] * x[2] * (A * B + C * D) - 2 * A * B * C * D
                one = 2 * ((A + D) * (A * D - B * C) - x[1] * x[1] * D - x[2] * x[2] * A + x[1] * x[2] *
                  (B + C))
                two = A * A + D * D + 4 * A * D - x[1] * x[1] - x[2] * x[2] - 2 * B * C
                three = 2 * (A + D)
                four = 1
                roots = polyroot(c(zero, one, two, three, four))
                return(roots)
            })
            ### Test each root with dual penalty system (parallelized) ###
            ze = foreach(j = 1:Ns, .combine = "cbind", .packages = c("base"), .export = c("apply_window_constraint",
                "calculate_window_penalty", "calculate_reference_penalty_sample", "current_decay_factor")) %dopar%
                {
                  possible = lapply(rooted[, j], function(y) {
                    if (abs(Im(y)) > 1e-08) {
                      return(c(0, 0, 1e+05, 1e+05))
                    }
                    K.lambda = K + diag(2) * Re(y)
                    zet = solve(K.lambda, O[, j])

                    phit = atan2(zet[2], zet[1])

                    # Apply soft window constraint if enabled
                    if (constrain_12h_window) {
                      phit <- apply_window_constraint(phit, window_start, window_width, window_prior_strength)
                    }

                    Xt = c(1, cos(phit), sin(phit))
                    Mt = Xt %o% Xt

                    Xt.old = c(1, cos(phiold[j]), sin(phiold[j]))
                    Mt.old = Xt.old %o% Xt.old

                    Qs = sapply(1:Ng, function(k) {
                      return(alphat[k, ] %*% Mt %*% alphat[k, ] - 2 * alphat[k, ] %*% Xt * E[k, j] +
                        sigma2 * sum(diag(Minv %*% Mt)))
                    })

                    Qs.old = sapply(1:Ng, function(k) {
                      return(alphat[k, ] %*% Mt.old %*% alphat[k, ] - 2 * alphat[k, ] %*% Xt.old * E[k,
                        j] + sigma2 * sum(diag(Minv %*% Mt.old)))
                    })

                    Q = sum(Qs * W/sigma2)/Tot
                    Q.old = sum(Qs.old * W/sigma2)/Tot
                    # Add window penalty if enabled
                    if (constrain_12h_window) {
                      window_penalty <- calculate_window_penalty(phit, window_start, window_width, window_prior_strength)
                      Q <- Q + window_penalty
                    }
                    # Add reference phase penalty if enabled (with decay)
                    if (!is.null(reference_genes) && !is.null(reference_phases)) {
                      # Apply decayed penalty strength
                      effective_strength <- reference_strength * current_decay_factor
                      reference_penalty <- calculate_reference_penalty_sample(phit, j, reference_gene_indices,
                        reference_phases, reference_troughs, E, alpha, effective_strength, use_trough_constraint)
                      Q <- Q + reference_penalty
                    }
                    return(c(zet, Q, Q.old))
                  })
                  pox = matrix(unlist(possible), 4, 4, byrow = T)
                  mpox = which.min(pox[, 3])
                  if (pox[mpox, 3] == 1e+05) {
                    cat("\n", pox, "not any solution on the circle at iteration", i, "\n")
                    stop()
                    return(list(phi = phiold, Qhist = Qhist, sigma = sigma2old, alpha = alpha, weights = W))
                  }
                  return(c(pox[mpox, ], pox[, 3]))
                }
            ### Update phases with dual constraints ### ze is already a matrix from foreach with
            ### .combine='cbind'
            rownames(ze) = c("cos", "sin", "Q", "Q.old", "Q1", "Q2", "Q3", "Q4")

            phi_new = atan2(ze[2, ], ze[1, ])%%(2 * pi)

            # Apply hard constraints post-update
            if (constrain_12h_window & post_hoc_hard_constrain) {
                phi_new <- constrain_phases_to_window(phi_new, window_start, window_width)
            }

            phi <- phi_new

            # Store iteration history with decay factor
            if (i == 1) {
                Qhist = as.data.frame(t(ze))
                colnames(Qhist) = c("cos", "sin", "Q", "Q.old", "Q1", "Q2", "Q3", "Q4")
                Qhist$iteration = i
                Qhist$sample = c(1:Ns)
                Qhist$decay_factor = current_decay_factor
            } else {
                Qtemp = data.frame(t(ze))
                colnames(Qtemp) = c("cos", "sin", "Q", "Q.old", "Q1", "Q2", "Q3", "Q4")
                Qtemp$iteration = i
                Qtemp$sample = c(1:Ns)
                Qtemp$decay_factor = current_decay_factor
                Qhist = rbind(Qhist, Qtemp)
            }

            # Update weights
            P1.0 = sapply(1:Ng, function(p) {
                gamm = sqrt(dTinv * sigma2^3/det(M))
                exponent = E[p, ] %*% X %*% Minv %*% t(X) %*% E[p, ]/(2 * sigma2)
                return(exp(exponent) * gamm)
            })
            W <- q * P1.0/(1 - q + q * P1.0)
            W[is.nan(W)] <- 1

            ### Check convergence ###
            if (max(abs(phi - phiold)) < (0.01)) {
                if (pbar) {
                  cat("\n algorithm has converged \n")
                }

                # Auto-adjust for potential anti-phase problem if reference genes are provided
                phi_adjusted <- phi
                if (auto_adjust_anti_phase && !is.null(reference_genes) && !is.null(reference_phases)) {
                  cat("\n### Post-hoc phase adjustment check ###\n")
                  phi_adjusted <- adjust_phase_by_reference(phi, alpha, E, reference_gene_indices, reference_phases,
                    reference_troughs, use_trough_constraint)
                }

                return(list(phi = phi_adjusted, phi_raw = phi, sigma = sigma2, alpha = alpha, weights = W,
                  iter = i, sigma.m1 = sigma2.m1, E = E.full, Qhist = Qhist, geni = geni, clock = E,
                  constrain_12h_window = constrain_12h_window, window_start = window_start, window_width = window_width,
                  reference_genes = reference_genes, reference_phases = reference_phases, reference_troughs = reference_troughs,
                  use_trough_constraint = use_trough_constraint, reference_strength = reference_strength,
                  auto_adjusted = !identical(phi, phi_adjusted), decay_history = decay_history[1:i],
                  reference_decay_enabled = reference_decay_enabled, decay_midpoint = decay_midpoint,
                  decay_sharpness = decay_sharpness))
            }

            ### Update parameters for next iteration ###
            c = cos(phi)
            s = sin(phi)
            one = rep(1, length(phi))
            X = cbind(one, c, s)

            Mold = Nn + sigma2 * Tinv
            Moldinv = solve(Mold)

            sigma2.m1 = sapply(1:Ng, function(s) {
                return(sum(diag(S[[s]] - S[[s]] %*% Xold %*% Moldinv %*% t(X)))/Ns + 0.01)
            })
            sigma2.m0 = apply(E, 1, var)
            sigma2 <- mean(sigma2.m1 * W + sigma2.m0 * (1 - W))
            sigma2.m1 = sum(sigma2.m1 * W)/sum(W)

            if (update.q) {
                q = mean(W)
                if (q < 0.05)
                  q = 0.05
                if (q > 0.3)
                  q = 0.3
            }
            if (pbar) {
                Sys.sleep(0.1)
                setTxtProgressBar(pb, i)
            }
        }

        if (pbar) {
            close(pb)
            cat("\n")
        }

        # Auto-adjust for potential anti-phase problem if reference genes are provided (max
        # iterations case)
        phi_adjusted <- phi
        if (auto_adjust_anti_phase && !is.null(reference_genes) && !is.null(reference_phases)) {
            cat("\n### Post-hoc phase adjustment check (max iterations reached) ###\n")
            phi_adjusted <- adjust_phase_by_reference(phi, alpha, E, reference_gene_indices, reference_phases,
                reference_troughs, use_trough_constraint)
        }

        return(list(phi = phi_adjusted, phi_raw = phi, Qhist = Qhist, sigma = sigma2, alpha = alpha,
            weights = W, iter = i, sigma.m1 = sigma2.m1, E = E.full, constrain_12h_window = constrain_12h_window,
            window_start = window_start, window_width = window_width, reference_genes = reference_genes,
            reference_phases = reference_phases, reference_troughs = reference_troughs, use_trough_constraint = use_trough_constraint,
            reference_strength = reference_strength, auto_adjusted = !identical(phi, phi_adjusted), decay_history = decay_history,
            reference_decay_enabled = reference_decay_enabled, decay_midpoint = decay_midpoint, decay_sharpness = decay_sharpness))
    }

    #' Calculate penalty for deviation from reference gene phases (peaks and troughs)
    #' @param phi_sample Current phase estimate for a sample
    #' @param sample_idx Index of current sample
    #' @param ref_gene_indices Indices of reference genes in the gene list
    #' @param ref_phases Expected peak phases for reference genes
    #' @param ref_troughs Expected trough phases for reference genes (can be NULL)
    #' @param E Expression matrix
    #' @param alpha Current gene parameters
    #' @param strength Penalty strength
    #' @param use_trough Whether to use trough constraint
    calculate_reference_penalty_sample <- function(phi_sample, sample_idx, ref_gene_indices, ref_phases,
        ref_troughs, E, alpha, strength, use_trough = TRUE) {
        if (is.null(ref_gene_indices) || length(ref_gene_indices) == 0) {
            return(0)
        }

        total_penalty <- 0

        for (i in 1:length(ref_gene_indices)) {
            gene_idx <- ref_gene_indices[i]
            expected_peak <- ref_phases[i]

            # Calculate when this gene would peak given current parameters
            a_coef <- alpha[gene_idx, "a"]
            b_coef <- alpha[gene_idx, "b"]

            # Gene peaks when derivative = 0: -a*sin(phi) + b*cos(phi) = 0 This gives: tan(phi) =
            # b/a, so phi = atan2(b, a)
            gene_peak_phase <- atan2(b_coef, a_coef)%%(2 * pi)

            # Gene troughs at peak + π
            gene_trough_phase <- (gene_peak_phase + pi)%%(2 * pi)

            # Calculate circular distance between expected and predicted peak
            peak_distance <- abs(((gene_peak_phase - expected_peak + pi)%%(2 * pi)) - pi)

            # Quadratic penalty weighted by gene expression level
            expr_weight <- abs(E[gene_idx, sample_idx]) + 0.1  # Avoid zero weight
            peak_penalty <- strength * peak_distance^2 * expr_weight
            total_penalty <- total_penalty + peak_penalty

            # Add trough constraint if enabled
            if (use_trough && !is.null(ref_troughs)) {
                expected_trough <- ref_troughs[i]
                trough_distance <- abs(((gene_trough_phase - expected_trough + pi)%%(2 * pi)) - pi)

                # Weight trough constraint slightly less than peak (0.7x) Peaks are usually more
                # reliable than troughs in gene expression data
                trough_penalty <- strength * 0.7 * trough_distance^2 * expr_weight
                total_penalty <- total_penalty + trough_penalty
            }
        }

        return(total_penalty)
    }

    #' Apply reference gene bias during initialization
    #' @param phi_init Initial phase estimates
    #' @param ref_gene_indices Indices of reference genes
    #' @param ref_phases Expected phases for reference genes
    #' @param E Expression matrix
    #' @param bias_strength Strength of bias (0-1)
    apply_reference_bias <- function(phi_init, ref_gene_indices, ref_phases, E, bias_strength = 0.3) {
        if (is.null(ref_gene_indices) || length(ref_gene_indices) == 0) {
            return(phi_init)
        }

        # For each sample, bias toward phases where reference genes would be expressed as expected
        for (s in 1:length(phi_init)) {
            reference_score <- 0
            total_weight <- 0

            for (i in 1:length(ref_gene_indices)) {
                gene_idx <- ref_gene_indices[i]
                expected_phase <- ref_phases[i]

                # Score how well current phase aligns with expected reference gene expression
                phase_diff <- abs(((phi_init[s] - expected_phase + pi)%%(2 * pi)) - pi)
                alignment_score <- cos(phase_diff)  # Higher when phases align

                expr_level <- abs(E[gene_idx, s]) + 0.1
                reference_score <- reference_score + alignment_score * expr_level
                total_weight <- total_weight + expr_level
            }

            if (total_weight > 0) {
                avg_reference_score <- reference_score/total_weight

                # If reference score is low, bias toward nearest reference phase
                if (avg_reference_score < 0.5) {
                  # Find nearest reference phase
                  distances <- sapply(ref_phases, function(rp) {
                    abs(((phi_init[s] - rp + pi)%%(2 * pi)) - pi)
                  })
                  nearest_ref_phase <- ref_phases[which.min(distances)]

                  # Bias toward nearest reference phase
                  phi_init[s] <- phi_init[s] * (1 - bias_strength) + nearest_ref_phase * bias_strength
                }
            }
        }

        return(phi_init%%(2 * pi))
    }

    #' Utility function to convert hours to phases
    #' @param hours Time in hours [0, 24)
    #' @return Phases in radians [0, 2π]
    hours_to_phases <- function(hours) {
        phi <- (hours/24) * 2 * pi
        return(phi%%(2 * pi))
    }

    #' Utility function to convert phases to hours
    #' @param phi Phases in radians [0, 2π]
    #' @return Time in hours [0, 24)
    phases_to_hours <- function(phi) {
        hours <- (phi/(2 * pi)) * 24
        return(hours)
    }


    # Backward compatibility functions - original CHIRAL functions kept unchanged

    #' Original CHIRAL function for backward compatibility
    CHIRAL <- function(E, iterations = 500, clockgenes = NULL, tau2 = NULL, u = NULL, sigma2 = NULL,
        TSM = TRUE, mean.centre.E = TRUE, q = 0.1, update.q = FALSE, pbar = TRUE, phi.start = NULL, standardize = FALSE,
        GTEx_names = FALSE) {

        # Call the new function with no window constraints for backward compatibility
        return(CHIRAL_dual_penalty(E, iterations = iterations, clockgenes = clockgenes, tau2 = tau2,
            u = u, sigma2 = sigma2, TSM = TSM, mean.centre.E = mean.centre.E, q = q, update.q = update.q,
            pbar = pbar, phi.start = phi.start, standardize = standardize, GTEx_names = GTEx_names, constrain_12h_window = FALSE))
    }

    cor.c <- function(x, y = NULL) {
        if (is.null(y)) {
            return(cor.c.g(x))
        }
        if (length(x) != length(y)) {
            cat("the two vectors have different length")
            return(NULL)
        }
        n = length(x)
        U1 = 0
        U2 = 0
        U3 = 0
        for (i in 1:(n - 1)) {
            for (j in ((i + 1):n)) {
                U1 = U1 + sin(x[i] - x[j]) * sin(y[i] - y[j])
                U2 = U2 + sin(x[i] - x[j])^2
                U3 = U3 + sin(y[i] - y[j])^2
            }
        }
        corr = U1/sqrt(U2 * U3)
        return(corr)
    }

    cor.c.g <- function(x) {
        off = ncol(x)
        cr = matrix(0, ncol = off, nrow = off)
        dimnames(cr) = list(colnames(x), colnames(x))
        for (l in 1:off) {
            for (m in l:off) {
                cr[l, m] = cr[m, l] = cor.c(x[, l], x[, m])
            }
        }
        return(cr)
    }

    delta.phi <- function(phi.0, phi, period = 2 * pi, mode = "forgotten", N = 200, median_scale = 1) {
        phi.0 = phi.0%%(period)
        phi = phi%%(period)
        isP = abs(period - 2 * pi) > 0.001
        if (!isP) {
            obj = calc.delta(phi.0, phi, N)
            bestphi = obj$phi
            mad = obj$median * median_scale
        } else {
            obj = calc.delta(phi.0/period * 2 * pi, phi/period * 2 * pi, N)
            bestphi = obj$phi * period/2/pi
            mad = obj$median * period/2/pi * median_scale
        }
        if (mode == "say") {
            cat("median:", mad, "\n")
            return(bestphi)
        } else if (mode == "return") {
            return(list(phi = bestphi, median = mad))
        } else if (mode == "no_median") {
            return(bestphi)
        } else {
            cat("median:", mad, "\n")
            return(list(phi = bestphi, median = (mad)))
        }
    }

    calc.delta <- function(phi.0, phi, N = 200) {
        mad <- 12
        sdel = NULL
        for (j in 1:N) {
            offset <- j/N * 2 * pi
            theta <- (phi - offset)%%(2 * pi)
            del <- abs(theta - phi.0)%%(2 * pi)
            delta <- del
            for (i in 1:length(phi)) {
                delta[i] = min(del[i], 2 * pi - del[i])
            }
            if (median(delta) < mad) {
                mad <- median(delta)
                bestphi <- theta
                j_temp = j
                sdel = delta
            }
            phi <- (-phi)%%(2 * pi)
            theta <- (phi - offset)%%(2 * pi)
            del <- abs(theta - phi.0)%%(2 * pi)
            delta <- del
            for (i in 1:length(phi)) {
                delta[i] = min(del[i], 2 * pi - del[i])
            }
            if (median(delta) < mad) {
                mad <- median(delta)
                bestphi <- theta
                j_temp = -j
                sdel = delta
            }
        }
        return(list(phi = bestphi, median = (mad)))
    }

    adjust.phases <- function(realphi, infphi, period = 2 * pi) {
        if (abs(period - 2 * pi) > 0.001) {
            return(adjust.phases(realphi * 2 * pi/period, infphi * 2 * pi/period) * period/pi/2)
        }
        for (i in 1:length(realphi)) {
            if (realphi[i] - infphi[i] > pi) {
                infphi[i] = 2 * pi + infphi[i]
            }
            if (realphi[i] - infphi[i] < (-pi)) {
                infphi[i] = infphi[i] - 2 * pi
            }
        }
        return(infphi)
    }

    ### Spin glass approximation to have initial condition for EM ###

    Zeta.mf.ordered <- function(J, beta, samples, A.0 = 0.1) {
        iterations <- 1000
        time_symmetry <- 0
        A <- rep(A.0, samples)
        Theta <- runif(samples, 0, 2 * pi)
        for (time in 1:iterations) {
            A.c = A * cos(Theta)
            A.s = A * sin(Theta)
            for (k in 1:samples) {
                u = beta * sum(A.c * J[k, ])
                v = beta * sum(A.s * J[k, ])

                modulo <- sqrt(u * u + v * v)
                Zeta_k <- c(u/modulo, v/modulo)
                Theta[k] <- atan2(Zeta_k[2], Zeta_k[1])
                A[k] <- (besselI(modulo, 1)/(besselI(modulo, 0)))
                if (is.na(modulo)) {
                  A[k] <- 1
                } else if (modulo > 20) {
                  A[k] <- 1
                }
                if (is.nan(u) || is.nan(v)) {
                  Theta[k] <- runif(1, 0, 2 * pi)
                } else if (u == 0) {
                  if (v == 0) {
                    Theta[k] <- runif(1, 0, 2 * pi)
                  }
                }
            }
        }
        for (k in 1:(samples - 1)) {
            if (Theta[k] < Theta[k + 1]) {
                time_symmetry <- time_symmetry + 1
            }
            if (Theta[k] > Theta[k + 1]) {
                time_symmetry <- time_symmetry - 1
            }
        }

        time_symmetry <- sign(time_symmetry)
        if (time_symmetry == -1) {
            for (k in 1:samples) {
                Theta[k] <- 2 * pi - Theta[k]
            }
        }
        Theta <- Theta%%(2 * pi)
        Zeta <- cbind(A, Theta)
        return(Zeta)
    }

    ### Calculation of spin glass interaction matrix ###

    J.tilde <- function(E, n.genes = 0, n.samples = 0) {
        sda <- as.matrix(E)
        if (n.genes == 0) {
            n.genes = nrow(E)
        }
        if (n.samples == 0) {
            n.samples = ncol(E)
        }
        Jtilde <- matrix(0, ncol = n.samples, nrow = n.samples)
        for (i in 1:n.samples) {
            for (j in 1:n.samples) {
                Jtilde[i, j] <- sum(sda[, i] * sda[, j])/(n.samples * n.genes)
            }
        }
        diag(Jtilde) <- 0
        rownames(Jtilde) <- colnames(E)
        colnames(Jtilde) <- colnames(E)
        return(Jtilde)
    }

    #' Apply soft window constraint during optimization
    apply_window_constraint <- function(phi, window_start, window_width, strength) {
        # Soft constraint - don't force hard boundaries during optimization Let the optimization
        # explore but bias toward window

        window_center <- (window_start + window_width/2)%%(2 * pi)

        # Calculate distance from window center
        dist_from_center <- abs(((phi - window_center + pi)%%(2 * pi)) - pi)

        # If outside acceptable range, bias toward window center
        max_allowed_dist <- window_width/2 * 1.2  # Allow some flexibility

        if (dist_from_center > max_allowed_dist) {
            # Bias toward window center
            bias_factor <- 0.8
            phi <- phi * (1 - bias_factor) + window_center * bias_factor
        }

        return(phi%%(2 * pi))
    }

    #' Calculate penalty for being outside sampling window
    calculate_window_penalty <- function(phi, window_start, window_width, strength) {
        window_end <- (window_start + window_width)%%(2 * pi)

        # Calculate distance outside window using circular distance
        if (window_end < window_start) {
            # Wrap-around case (e.g., 10PM to 10AM)
            in_window <- (phi >= window_start) | (phi <= window_end)
            if (in_window)
                return(0)

            # Calculate circular distance to nearest window boundary
            dist_to_start <- pmin(abs(phi - window_start), abs(phi - window_start + 2 * pi), abs(phi -
                window_start - 2 * pi))
            dist_to_end <- pmin(abs(phi - window_end), abs(phi - window_end + 2 * pi), abs(phi - window_end -
                2 * pi))
            dist_out <- pmin(dist_to_start, dist_to_end)
        } else {
            # Normal case (e.g., 6AM to 6PM) Check if phi is in window using circular logic
            if (phi >= window_start && phi <= window_end)
                return(0)

            # Calculate circular distance to nearest window boundary
            dist_to_start <- pmin(abs(phi - window_start), abs(phi - window_start + 2 * pi), abs(phi -
                window_start - 2 * pi))
            dist_to_end <- pmin(abs(phi - window_end), abs(phi - window_end + 2 * pi), abs(phi - window_end -
                2 * pi))
            dist_out <- pmin(dist_to_start, dist_to_end)
        }

        # Quadratic penalty
        penalty <- strength * dist_out^2
        return(penalty)
    }

    #' Post-hoc phase adjustment to fix potential π-shift (anti-phase) problem
    #'
    #' This function checks EACH SAMPLE independently to see if it should be shifted by π
    #' (12h) to better align with reference gene expectations.
    #'
    #' @param phi Inferred phases in radians [0, 2π]
    #' @param alpha Gene coefficient matrix (Ng x 3, columns: mu, a, b)
    #' @param E Expression matrix (genes x samples)
    #' @param ref_gene_indices Indices of reference genes
    #' @param ref_phases Expected peak phases for reference genes
    #' @param ref_troughs Expected trough phases for reference genes (can be NULL)
    #' @param use_trough Whether to use trough constraint
    #' @return Adjusted phases that better align with reference genes (per-sample decision)
    #' @examples
    #' # After running CHIRAL_dual_penalty:
    #' result <- CHIRAL_dual_penalty(data, reference_genes=..., reference_phases=...)
    #' adjusted_phi <- adjust_phase_by_reference(
    #'   result$phi, result$alpha, result$clock,
    #'   ref_gene_indices, ref_phases, ref_troughs
    #' )
    adjust_phase_by_reference <- function(phi, alpha, E, ref_gene_indices, ref_phases, ref_troughs = NULL,
        use_trough = TRUE) {

        if (is.null(ref_gene_indices) || length(ref_gene_indices) == 0) {
            message("No reference genes provided. Returning original phases.")
            return(phi)
        }

        n_samples <- length(phi)
        phi_adjusted <- phi
        n_adjusted <- 0

        # Evaluate each sample independently
        for (s in 1:n_samples) {
            phi_original <- phi[s]
            phi_shifted <- (phi[s] + pi)%%(2 * pi)

            # Calculate alignment scores for this sample
            score_original <- calculate_sample_reference_score(phi_original, s, alpha, E, ref_gene_indices,
                ref_phases, ref_troughs, use_trough)

            score_shifted <- calculate_sample_reference_score(phi_shifted, s, alpha, E, ref_gene_indices,
                ref_phases, ref_troughs, use_trough)

            # Choose the version with better alignment for THIS sample
            if (score_shifted < score_original) {
                phi_adjusted[s] <- phi_shifted
                n_adjusted <- n_adjusted + 1
            }
        }

        # Report per-sample results
        if (n_adjusted > 0) {
            pct_adjusted <- (n_adjusted/n_samples) * 100
            message(sprintf("[Per-sample] Adjusted %d/%d samples (%.1f%%) by π-shift", n_adjusted, n_samples,
                pct_adjusted))
        } else {
            message("[Per-sample] No individual samples required adjustment")
        }

        # NOW: Check if shifting ALL samples together is even better (global adjustment)
        message("\n[Global check] Testing if shifting all samples by π improves overall alignment...")

        phi_after_individual <- phi_adjusted
        phi_global_shifted <- (phi_adjusted + pi)%%(2 * pi)

        # Calculate total scores for both options
        score_individual <- 0
        score_global_shift <- 0

        for (s in 1:n_samples) {
            score_individual <- score_individual + calculate_sample_reference_score(phi_after_individual[s],
                s, alpha, E, ref_gene_indices, ref_phases, ref_troughs, use_trough)
            score_global_shift <- score_global_shift + calculate_sample_reference_score(phi_global_shifted[s],
                s, alpha, E, ref_gene_indices, ref_phases, ref_troughs, use_trough)
        }

        # Choose the better option
        if (score_global_shift < score_individual) {
            improvement <- ((score_individual - score_global_shift)/score_individual) * 100
            message(sprintf("[Global] Shifting ALL samples by π improves alignment by %.1f%%. Applying global shift.",
                improvement))
            message(sprintf("[Global] Score after individual adjustment: %.3f, Score after global shift: %.3f",
                score_individual, score_global_shift))
            return(phi_global_shifted)
        } else {
            message(sprintf("[Global] No global shift needed. Individual adjustments are optimal."))
            message(sprintf("[Global] Score after individual adjustment: %.3f, Score if globally shifted: %.3f",
                score_individual, score_global_shift))
            return(phi_after_individual)
        }
    }

    #' Calculate reference alignment score for a SINGLE sample
    #'
    #' Helper function to score how well a sample's reference gene expression
    #' aligns with expected peak/trough patterns at a given phase.
    #'
    #' @param phi_sample Phase to evaluate for this sample
    #' @param sample_idx Index of the sample
    #' @param alpha Gene coefficients
    #' @param E Expression matrix
    #' @param ref_gene_indices Indices of reference genes
    #' @param ref_phases Expected peak phases
    #' @param ref_troughs Expected trough phases (can be NULL)
    #' @param use_trough Whether to use trough constraint
    #' @return Misalignment score for this sample (lower = better)
    calculate_sample_reference_score <- function(phi_sample, sample_idx, alpha, E, ref_gene_indices,
        ref_phases, ref_troughs, use_trough) {
        total_score <- 0

        for (i in 1:length(ref_gene_indices)) {
            gene_idx <- ref_gene_indices[i]
            expected_peak <- ref_phases[i]

            # Get gene coefficients
            a_coef <- alpha[gene_idx, "a"]
            b_coef <- alpha[gene_idx, "b"]
            mu_coef <- alpha[gene_idx, "mu"]

            # Calculate expected expression at this phase
            expected_expr <- mu_coef + a_coef * cos(phi_sample) + b_coef * sin(phi_sample)

            # Calculate expected expression at peak phase
            peak_expr <- mu_coef + a_coef * cos(expected_peak) + b_coef * sin(expected_peak)

            # Actual expression of this gene in this sample
            actual_expr <- E[gene_idx, sample_idx]

            # Score based on how well actual expression matches what we'd expect If sample is at
            # expected peak time, gene should be highly expressed This measures circular
            # correlation between phase and expression
            gene_peak_phase <- atan2(b_coef, a_coef)%%(2 * pi)
            phase_distance_to_peak <- abs(((phi_sample - gene_peak_phase + pi)%%(2 * pi)) - pi)

            # Score: gene should peak when reference says it should
            peak_time_distance <- abs(((gene_peak_phase - expected_peak + pi)%%(2 * pi)) - pi)

            # Weight by expression amplitude
            amplitude <- sqrt(a_coef^2 + b_coef^2)
            total_score <- total_score + peak_time_distance^2 * (amplitude + 0.1)

            # Add trough constraint if enabled
            if (use_trough && !is.null(ref_troughs)) {
                expected_trough <- ref_troughs[i]
                gene_trough_phase <- (gene_peak_phase + pi)%%(2 * pi)
                trough_time_distance <- abs(((gene_trough_phase - expected_trough + pi)%%(2 * pi)) -
                  pi)
                total_score <- total_score + 0.7 * trough_time_distance^2 * (amplitude + 0.1)
            }
        }

        return(total_score)
    }

    #' Initialize phases based on reference gene expression patterns
    #'
    #' This function uses reference gene expression levels to directly infer
    #' initial phase estimates, avoiding the anti-phase trap. When window constraints
    #' are provided, it restricts candidate phases to the valid sampling window.
    #'
    #' @param E Expression matrix (genes x samples)
    #' @param ref_gene_indices Indices of reference genes
    #' @param ref_phases Expected peak phases for reference genes
    #' @param ref_troughs Expected trough phases (optional)
    #' @param constrain_window Whether to constrain to sampling window (default FALSE)
    #' @param window_start Start of sampling window in radians (default 0)
    #' @param window_width Width of sampling window in radians (default 2π)
    #' @param fallback_init Fallback initialization if reference-based fails
    #' @return Vector of initial phase estimates [0, 2π]
    #' @examples
    #' # Without window constraint
    #' phi_init <- initialize_phases_by_reference(
    #'   E, ref_gene_indices, ref_phases, ref_troughs
    #' )
    #'
    #' # With window constraint (e.g., daytime sampling only)
    #' phi_init <- initialize_phases_by_reference(
    #'   E, ref_gene_indices, ref_phases, ref_troughs,
    #'   constrain_window = TRUE, window_start = 0, window_width = pi
    #' )
    initialize_phases_by_reference <- function(E, ref_gene_indices, ref_phases, ref_troughs = NULL, constrain_window = FALSE,
        window_start = 0, window_width = 2 * pi, fallback_init = NULL) {

        n_samples <- ncol(E)
        phi_init <- numeric(n_samples)

        # Generate candidate phases
        if (constrain_window && window_width < 2 * pi) {
            # IMPORTANT: If window constraint is enabled, only test phases WITHIN the window This
            # represents the biological reality: samples were collected during a limited time E.g.,
            # patient samples collected only during clinic hours (8am-8pm)

            message(sprintf("Constraining initialization to window [%.2f, %.2f] radians (%.1f-%.1f hours)",
                window_start, (window_start + window_width)%%(2 * pi), window_start * 12/pi, ((window_start +
                  window_width)%%(2 * pi)) * 12/pi))

            # Generate candidates within window only (every 15 degrees)
            n_candidates <- max(8, ceiling(window_width/(pi/12)))  # At least 8 candidates
            candidate_phases <- seq(window_start, window_start + window_width, length.out = n_candidates)
            # NOTE: Do NOT apply modulo here - keep phases in their original range This preserves
            # proper ordering for windows that may extend beyond 2π
        } else {
            # Test full 24-hour range (every 15 degrees = π/12 = 1 hour)
            candidate_phases <- seq(0, 2 * pi - 0.01, by = pi/12)
        }

        # Strategy: For each sample, find the phase where reference genes would have expression
        # patterns most similar to observed

        for (s in 1:n_samples) {
            # Get expression levels of reference genes in this sample
            ref_expr <- E[ref_gene_indices, s]

            best_score <- Inf
            best_phase <- 0

            for (phi_test in candidate_phases) {
                score <- 0

                # For each reference gene, calculate expected vs observed expression similarity
                for (i in 1:length(ref_gene_indices)) {
                  expected_peak <- ref_phases[i]
                  actual_expr <- ref_expr[i]

                  # Expected expression pattern: high near peak, low near trough Use cosine
                  # similarity to expected peak time
                  peak_distance <- abs(((phi_test - expected_peak + pi)%%(2 * pi)) - pi)

                  # Expected relative expression (0 to 1 scale) When φ = expected_peak:
                  # expected_rel_expr = 1 When φ = expected_peak ± π: expected_rel_expr = 0
                  expected_rel_expr <- (cos(peak_distance) + 1)/2

                  # Normalize actual expression to [0, 1] using percentile rank This handles
                  # different gene expression levels
                  gene_expr_all_samples <- E[ref_gene_indices[i], ]
                  actual_percentile <- sum(gene_expr_all_samples <= actual_expr)/n_samples

                  # Score is squared difference between expected and observed
                  score <- score + (expected_rel_expr - actual_percentile)^2

                  # Add trough constraint if provided
                  if (!is.null(ref_troughs)) {
                    expected_trough <- ref_troughs[i]
                    trough_distance <- abs(((phi_test - expected_trough + pi)%%(2 * pi)) - pi)
                    expected_rel_expr_trough <- (cos(trough_distance) + 1)/2

                    # At trough, expression should be LOW (percentile should be low) So we want:
                    # when expected_rel_expr_trough is high → actual should be low
                    score <- score + 0.5 * (expected_rel_expr_trough - (1 - actual_percentile))^2
                  }
                }

                # Choose phase with best (lowest) score
                if (score < best_score) {
                  best_score <- score
                  best_phase <- phi_test
                }
            }

            phi_init[s] <- best_phase
        }

        # Add small random noise to avoid exact duplicates
        phi_init <- phi_init + runif(n_samples, -0.1, 0.1)
        phi_init <- phi_init%%(2 * pi)

        # Validate initialization
        if (any(is.na(phi_init)) || any(is.nan(phi_init))) {
            warning("Reference-based initialization failed. Using fallback.")
            if (!is.null(fallback_init)) {
                return(fallback_init)
            } else {
                return(runif(n_samples, 0, 2 * pi))
            }
        }

        message(sprintf("Initialized phases using reference gene expression patterns (range: %.2f to %.2f radians)",
            min(phi_init), max(phi_init)))

        return(phi_init)
    }

    #' Helper function to constrain phases to sampling window
    #' @param phi Vector of phases in [0, 2π]
    #' @param window_start Start of window in radians
    #' @param window_width Width of window in radians
    #' @return Constrained phases
    constrain_phases_to_window <- function(phi, window_start, window_width) {
        window_end <- (window_start + window_width)%%(2 * pi)

        # Handle wrap-around case (e.g., 10PM to 10AM)
        if (window_end < window_start) {
            # Window wraps around midnight
            in_window <- (phi >= window_start) | (phi <= window_end)
            phi[!in_window] <- find_nearest_window_phase(phi[!in_window], window_start, window_end, wrap = TRUE)
        } else {
            # Normal case (e.g., 6AM to 6PM) BUT: Special handling when window_start is near 0 (<
            # 0.1 radians) Phases near 2π should be treated as equivalent to 0
            if (window_start < 0.1) {
                # For phases very close to 2π, check if they're circularly within the window E.g.,
                # if window is [0, π] and φ=6.2, check if (φ - 2π) would be in window
                phi_normalized <- phi%%(2 * pi)
                alternative_phi <- ifelse(phi_normalized > 1.5 * pi, phi_normalized - 2 * pi, phi_normalized)
                in_window <- ((phi_normalized >= window_start) & (phi_normalized <= window_end)) | ((alternative_phi >=
                  window_start) & (alternative_phi <= window_end))
            } else {
                in_window <- (phi >= window_start) & (phi <= window_end)
            }
            phi[!in_window] <- find_nearest_window_phase(phi[!in_window], window_start, window_end, wrap = FALSE)
        }

        return(phi)
    }


    #' Find nearest phase within the sampling window
    find_nearest_window_phase <- function(phi, window_start, window_end, wrap = FALSE) {
        # ALWAYS use circular distance, even for non-wrapping windows This is critical when
        # window_start=0, as phases near 2π are very close to 0

        # Circular distance calculation
        dist_to_start <- pmin(abs(phi - window_start), abs(phi - window_start + 2 * pi), abs(phi - window_start -
            2 * pi))
        dist_to_end <- pmin(abs(phi - window_end), abs(phi - window_end + 2 * pi), abs(phi - window_end -
            2 * pi))

        nearest <- ifelse(dist_to_start < dist_to_end, window_start, window_end)

        return(nearest)
    }
}, envir = legacy_chiral)
pre_fix_chiral <- legacy_chiral
stopifnot(is.function(legacy_chiral$CHIRAL_dual_penalty))



In [ ]:
# Cell 4: Paired benchmark inference and scoring
benchmark_plan <- tidyr::crossing(dataset = names(benchmark_datasets), window = window_specs$window,
    method = STABILITY_METHODS, seed = STABILITY_SEEDS) |>
    dplyr::mutate(seed = as.integer(seed), rng_seed = RNG_SEED_BASE + seed) |>
    dplyr::left_join(window_specs, by = "window") |>
    dplyr::left_join(method_specs, by = "method")
stopifnot(nrow(benchmark_plan) == 5L * 5L * 2L * N_SEEDS)
run_chiral_one <- function(ds, row) {
    info <- ds$sample_info |>
        dplyr::filter(in_circular_window(timepoint_h, row$start_h, row$width_h), sample_id %in% colnames(ds$expr))
    E <- ds$expr[, info$sample_id, drop = FALSE]
    E <- E[rowSums(is.na(E)) < ncol(E), , drop = FALSE]
    set.seed(row$rng_seed)
    args <- list(E = E, iterations = ITERATIONS, clockgenes = ds$clock_genes, pbar = FALSE, n_cores = N_CORES)
    env <- legacy_chiral
    if (row$method == "modified_pre_fix_peak_trough") {
        env <- pre_fix_chiral
        args$reference_genes <- ds$ref_genes
        args$reference_phases <- ds$ref_phases
        args$reference_strength <- 1
        args$use_trough_constraint <- TRUE
        args$auto_adjust_anti_phase <- TRUE
        if (row$width_h < 24) {
            args$constrain_12h_window <- TRUE
            args$window_start <- hours_to_rad(row$start_h)
            args$window_width <- hours_to_rad(row$width_h)
        }
    }
    x <- tryCatch(do.call(env$CHIRAL_dual_penalty, args), error = function(e) e)
    common <- tibble::tibble(dataset = ds$id, window = row$window, method = row$method, seed = row$seed,
        rng_seed = row$rng_seed, sample_id = info$sample_id, group = info$group, truth_h = info$timepoint_h)
    if (inherits(x, "error"))
        return(dplyr::mutate(common, predicted_h = NA_real_, status = "error", error_message = conditionMessage(x),
            iter = NA_integer_, auto_adjusted = NA))
    dplyr::mutate(common, predicted_h = rad_to_hours(x$phi), status = "ok", error_message = NA_character_,
        iter = as.integer(x$iter %||% x$iteration %||% NA_integer_), auto_adjusted = as.logical(x$auto_adjusted %||%
            NA))
}
prediction_file <- file.path(BENCHMARK_DIR, "chiral_1_6_3_predictions.csv")
legacy_prediction_file <- file.path(LEGACY_BENCHMARK_DIR, "chiral_1_6_3_predictions.csv")
if (isTRUE(RUN_ANALYSIS)) {
    dir.create(BENCHMARK_DIR, recursive = TRUE, showWarnings = FALSE)
    benchmark_predictions <- dplyr::bind_rows(lapply(seq_len(nrow(benchmark_plan)), function(i) run_chiral_one(benchmark_datasets[[benchmark_plan$dataset[[i]]]],
        benchmark_plan[i, ])))
    stopifnot(all(benchmark_predictions$status == "ok"))
    readr::write_csv(benchmark_predictions, prediction_file)
} else benchmark_predictions <- readr::read_csv(legacy_prediction_file, show_col_types = FALSE) |>
    dplyr::filter(method %in% STABILITY_METHODS, seed %in% STABILITY_SEEDS)
# Cache mode preserves the audited best-global transform after CSV round-tripping raw predictions.
# Refit mode calculates the transform directly from in-memory predictions.
score_raw <- function(df) dplyr::mutate(df, alignment = "raw", predicted_eval_h = predicted_h%%24, alignment_sign = 1,
    alignment_offset_h = 0, signed_error_h = circular_signed_error_h(predicted_eval_h, truth_h), abs_error_h = abs(signed_error_h))
if (isTRUE(RUN_ANALYSIS)) {
    scored_predictions <- benchmark_predictions |>
        dplyr::filter(status == "ok", is.finite(predicted_h), is.finite(truth_h)) |>
        dplyr::group_by(dataset, window, method, seed) |>
        dplyr::group_modify(~score_prediction_group(.x)) |>
        dplyr::ungroup()
    alignment_transform_audit <- scored_predictions |>
        dplyr::filter(alignment == "best_global") |>
        dplyr::group_by(dataset, window, method, seed) |>
        dplyr::summarise(stored_alignment_sign = first(alignment_sign), stored_alignment_offset_h = first(alignment_offset_h),
            fresh_median_abs_error_h = median(abs_error_h), stored_median_abs_error_h = fresh_median_abs_error_h,
            transform_differs = FALSE, .groups = "drop")
} else {
    legacy_scored_file <- file.path(LEGACY_BENCHMARK_DIR, "chiral_1_6_3_scored_predictions.csv")
    legacy_scored <- readr::read_csv(legacy_scored_file, show_col_types = FALSE) |>
        dplyr::filter(method %in% STABILITY_METHODS, seed %in% STABILITY_SEEDS, status == "ok")
    transform_keys <- c("dataset", "window", "method", "seed")
    stored_transform <- legacy_scored |>
        dplyr::filter(alignment == "best_global") |>
        dplyr::distinct(dplyr::across(dplyr::all_of(transform_keys)), alignment_sign, alignment_offset_h)
    stopifnot(all(stored_transform |>
        dplyr::count(dplyr::across(dplyr::all_of(transform_keys))) |>
        dplyr::pull(n) == 1L))
    raw_predictions <- benchmark_predictions |>
        dplyr::filter(status == "ok", is.finite(predicted_h), is.finite(truth_h))
    raw_scored <- score_raw(raw_predictions)
    aligned_scored <- raw_predictions |>
        dplyr::inner_join(stored_transform, by = transform_keys) |>
        dplyr::mutate(alignment = "best_global", predicted_eval_h = (alignment_sign * predicted_h + alignment_offset_h)%%24,
            signed_error_h = circular_signed_error_h(predicted_eval_h, truth_h), abs_error_h = abs(signed_error_h))
    scored_predictions <- dplyr::bind_rows(raw_scored, aligned_scored)
    fresh_transform <- raw_predictions |>
        dplyr::group_by(dataset, window, method, seed) |>
        dplyr::group_modify(~score_prediction_group(.x)) |>
        dplyr::filter(alignment == "best_global") |>
        dplyr::group_by(dataset, window, method, seed) |>
        dplyr::summarise(fresh_alignment_sign = first(alignment_sign), fresh_alignment_offset_h = first(alignment_offset_h),
            fresh_median_abs_error_h = median(abs_error_h), .groups = "drop")
    stored_error <- aligned_scored |>
        dplyr::group_by(dataset, window, method, seed) |>
        dplyr::summarise(stored_median_abs_error_h = median(abs_error_h), .groups = "drop")
    alignment_transform_audit <- stored_transform |>
        dplyr::rename(stored_alignment_sign = alignment_sign, stored_alignment_offset_h = alignment_offset_h) |>
        dplyr::left_join(fresh_transform, by = transform_keys) |>
        dplyr::left_join(stored_error, by = transform_keys) |>
        dplyr::mutate(transform_differs = stored_alignment_sign != fresh_alignment_sign | abs(stored_alignment_offset_h -
            fresh_alignment_offset_h) > 1e-12, median_error_difference_h = stored_median_abs_error_h -
            fresh_median_abs_error_h)
    stopifnot(all(abs(alignment_transform_audit$median_error_difference_h) <= 1e-08))
}
scored_predictions <- scored_predictions |>
    dplyr::left_join(window_specs, by = "window") |>
    dplyr::left_join(method_specs |>
        dplyr::select(method, method_label, truth_during_inference, deployable_absolute_phase), by = "method") |>
    dplyr::left_join(alignment_specs, by = "alignment")
benchmark_run_summary <- scored_predictions |>
    dplyr::group_by(dataset, window, window_label, method, method_label, alignment, alignment_label,
        seed, rng_seed) |>
    dplyr::summarise(n_samples = dplyr::n_distinct(sample_id), run_mean_abs_error_h = mean(abs_error_h,
        na.rm = TRUE), run_median_abs_error_h = median(abs_error_h, na.rm = TRUE), run_rmse_h = sqrt(mean(signed_error_h^2,
        na.rm = TRUE)), run_bias_h = mean(signed_error_h, na.rm = TRUE), run_within_2h = mean(abs_error_h <=
        2, na.rm = TRUE), run_within_4h = mean(abs_error_h <= 4, na.rm = TRUE), run_severe_12h_flip_rate = mean(abs_error_h >=
        10, na.rm = TRUE), run_median_iter = median(iter, na.rm = TRUE), .groups = "drop")
benchmark_summary <- benchmark_run_summary |>
    dplyr::group_by(dataset, window, window_label, method, method_label, alignment, alignment_label) |>
    dplyr::summarise(n_samples = max(n_samples), n_runs = dplyr::n_distinct(seed), mean_abs_error_h = median(run_mean_abs_error_h,
        na.rm = TRUE), median_abs_error_h = median(run_median_abs_error_h, na.rm = TRUE), median_abs_error_q25_h = quantile(run_median_abs_error_h,
        0.25, na.rm = TRUE, names = FALSE), median_abs_error_q75_h = quantile(run_median_abs_error_h,
        0.75, na.rm = TRUE, names = FALSE), median_abs_error_min_h = min(run_median_abs_error_h, na.rm = TRUE),
        median_abs_error_max_h = max(run_median_abs_error_h, na.rm = TRUE), rmse_h = median(run_rmse_h,
            na.rm = TRUE), bias_h = median(run_bias_h, na.rm = TRUE), within_2h = median(run_within_2h,
            na.rm = TRUE), within_4h = median(run_within_4h, na.rm = TRUE), severe_12h_flip_rate = median(run_severe_12h_flip_rate,
            na.rm = TRUE), median_iter = median(run_median_iter, na.rm = TRUE), .groups = "drop")
readr::write_csv(scored_predictions, file.path(BENCHMARK_DIR, "chiral_1_6_3_scored_predictions.csv"))
readr::write_csv(benchmark_run_summary, file.path(BENCHMARK_DIR, "chiral_1_6_3_run_summary.csv"))
readr::write_csv(benchmark_summary, file.path(BENCHMARK_DIR, "chiral_1_6_3_summary.csv"))
readr::write_csv(alignment_transform_audit, file.path(BENCHMARK_DIR, "chiral_1_6_3_alignment_transform_audit.csv"))



In [ ]:
# Cell 5: Paired improvement and cross-seed concordance
keys <- c("dataset", "window", "seed", "sample_id", "alignment")
paired_rows <- dplyr::inner_join(scored_predictions |>
    dplyr::filter(method == "original_default") |>
    dplyr::select(dplyr::all_of(keys), baseline_abs_error_h = abs_error_h), scored_predictions |>
    dplyr::filter(method == "modified_pre_fix_peak_trough") |>
    dplyr::select(dplyr::all_of(keys), challenger_abs_error_h = abs_error_h), by = keys) |>
    dplyr::mutate(baseline = "original_default", challenger = "modified_pre_fix_peak_trough", contrast = "modified_pre_fix_peak_trough vs original_default",
        delta_abs_error_h = baseline_abs_error_h - challenger_abs_error_h)
paired_improvement_by_seed <- paired_rows |>
    dplyr::group_by(contrast, baseline, challenger, dataset, window, alignment, seed) |>
    dplyr::summarise(n_pairs = dplyr::n(), n_improved = sum(delta_abs_error_h > 0), n_worse = sum(delta_abs_error_h <
        0), mean_delta_abs_error_h = mean(delta_abs_error_h), median_delta_abs_error_h = median(delta_abs_error_h),
        .groups = "drop")
paired_improvement <- paired_improvement_by_seed |>
    dplyr::group_by(contrast, baseline, challenger, dataset, window, alignment) |>
    dplyr::summarise(n_runs = dplyr::n_distinct(seed), median_run_mean_delta_h = median(mean_delta_abs_error_h),
        run_mean_delta_q25_h = quantile(mean_delta_abs_error_h, 0.25, names = FALSE), run_mean_delta_q75_h = quantile(mean_delta_abs_error_h,
            0.75, names = FALSE), fraction_runs_mean_improved = mean(mean_delta_abs_error_h > 0), median_run_median_delta_h = median(median_delta_abs_error_h),
        .groups = "drop")
concordance_to_seed1 <- function(df) {
    ref <- df |>
        dplyr::filter(seed == 1L) |>
        dplyr::select(sample_id, reference_predicted_h = predicted_h)
    purrr::map_dfr(2:N_SEEDS, function(s) {
        y <- dplyr::inner_join(ref, df |>
            dplyr::filter(seed == s) |>
            dplyr::select(sample_id, target_predicted_h = predicted_h), by = "sample_id")
        fit <- fit_best_circular_alignment(y$target_predicted_h, y$reference_predicted_h, TRUE)
        tibble::tibble(reference_seed = 1L, target_seed = s, n_samples = nrow(y), raw_median_run_distance_h = median(circular_distance_h(y$target_predicted_h,
            y$reference_predicted_h)), symmetry_adjusted_median_run_distance_h = median(circular_distance_h((fit$sign *
            y$target_predicted_h + fit$offset_h)%%24, y$reference_predicted_h)), alignment_sign = fit$sign,
            alignment_offset_h = fit$offset_h)
    })
}
run_concordance <- scored_predictions |>
    dplyr::filter(alignment == "raw") |>
    dplyr::group_by(dataset, window, method) |>
    dplyr::group_modify(~concordance_to_seed1(.x)) |>
    dplyr::ungroup()
seed_stability_overall_paired <- benchmark_run_summary |>
    dplyr::group_by(alignment, method, seed) |>
    dplyr::summarise(mean_cell_median_error_h = mean(run_median_abs_error_h), .groups = "drop") |>
    tidyr::pivot_wider(names_from = method, values_from = mean_cell_median_error_h) |>
    dplyr::mutate(delta_original_minus_modified_h = original_default - modified_pre_fix_peak_trough)
seed_stability_overall_summary <- seed_stability_overall_paired |>
    dplyr::group_by(alignment) |>
    dplyr::summarise(n_paired_seeds = dplyr::n(), median_original_error_h = median(original_default),
        median_modified_error_h = median(modified_pre_fix_peak_trough), median_delta_h = median(delta_original_minus_modified_h),
        delta_q25_h = quantile(delta_original_minus_modified_h, 0.25, names = FALSE), delta_q75_h = quantile(delta_original_minus_modified_h,
            0.75, names = FALSE), fraction_seeds_modified_better = mean(delta_original_minus_modified_h >
            0), .groups = "drop")
run_concordance_summary <- run_concordance |>
    dplyr::group_by(dataset, window, method) |>
    dplyr::summarise(n_comparisons_to_seed1 = dplyr::n(), median_raw_run_distance_h = median(raw_median_run_distance_h),
        q75_raw_run_distance_h = quantile(raw_median_run_distance_h, 0.75, names = FALSE), max_raw_run_distance_h = max(raw_median_run_distance_h),
        median_symmetry_adjusted_run_distance_h = median(symmetry_adjusted_median_run_distance_h), q75_symmetry_adjusted_run_distance_h = quantile(symmetry_adjusted_median_run_distance_h,
            0.75, names = FALSE), max_symmetry_adjusted_run_distance_h = max(symmetry_adjusted_median_run_distance_h),
        .groups = "drop")
readr::write_csv(paired_rows, file.path(BENCHMARK_DIR, "chiral_1_6_3_paired_sample_deltas.csv"))
readr::write_csv(paired_improvement_by_seed, file.path(BENCHMARK_DIR, "chiral_1_6_3_paired_improvement_by_seed.csv"))
readr::write_csv(paired_improvement, file.path(BENCHMARK_DIR, "chiral_1_6_3_paired_improvement.csv"))
readr::write_csv(seed_stability_overall_paired, file.path(BENCHMARK_DIR, "chiral_1_6_3_seed_stability_overall_paired.csv"))
readr::write_csv(seed_stability_overall_summary, file.path(BENCHMARK_DIR, "chiral_1_6_3_seed_stability_overall_summary.csv"))
readr::write_csv(run_concordance, file.path(BENCHMARK_DIR, "chiral_1_6_3_run_concordance_to_seed1.csv"))
readr::write_csv(run_concordance_summary, file.path(BENCHMARK_DIR, "chiral_1_6_3_run_concordance_summary.csv"))



In [ ]:
# Cell 6: Sampling-window perturbation analysis
POINT5_OUTPUT_DIR <- POINT5_DIR
log_message <- function(...) message(sprintf(...))
POINT5_SEEDS <- seq_len(N_SEEDS)
N_OUTER_WORKERS <- 1L
RESUME_MATCHING_RUNS <- TRUE
FORCE_RERUN <- FALSE
source_windows <- tribble(~source_window, ~actual_start_h, ~actual_width_h, ~source_window_label, "0_12h",
    0, 12, "0-12 h", "6_18h", 6, 12, "6-18 h", "12_24h", 12, 12, "12-24 h", "18_6h", 18, 12, "18-6 h")

condition_specs <- tribble(~condition, ~condition_order, ~condition_kind, ~fixed_shift_h, ~window_enabled,
    ~condition_label, "correct_window", 1L, "correct", 0, TRUE, "Correct supplied window", "window_off",
    2L, "off", NA_real_, FALSE, "Window disabled", "shift_minus_4h", 3L, "shifted", -4, TRUE, "Window shifted -4 h",
    "shift_minus_2h", 4L, "shifted", -2, TRUE, "Window shifted -2 h", "shift_plus_2h", 5L, "shifted",
    2, TRUE, "Window shifted +2 h", "shift_plus_4h", 6L, "shifted", 4, TRUE, "Window shifted +4 h", "shift_plus_6h",
    7L, "shifted", 6, TRUE, "Window shifted +6 h", "shift_plus_12h", 8L, "shifted", 12, TRUE, "Window shifted +12 h",
    "permuted_window_labels", 9L, "permuted", NA_real_, TRUE, "Prespecified permutation of window labels")

# A cohort-level derangement: every actual window receives a different one of the four benchmark
# window labels, while the multiset of supplied starts is preserved. This is the meaningful
# analogue of a shuffled-window control for a method that takes one shared cohort window rather
# than sample-specific ones.
permuted_start_by_window <- c(`0_12h` = 12, `6_18h` = 18, `12_24h` = 6, `18_6h` = 0)
stopifnot(setequal(unname(permuted_start_by_window), source_windows$actual_start_h), all(unname(permuted_start_by_window[source_windows$source_window]) !=
    source_windows$actual_start_h))

signed_circular_shift_h <- function(supplied_start_h, actual_start_h) {
    ((supplied_start_h - actual_start_h + 12)%%24) - 12
}

point5_plan <- tidyr::crossing(dataset = names(benchmark_datasets), source_window = source_windows$source_window,
    condition = condition_specs$condition, seed = POINT5_SEEDS) %>%
    left_join(source_windows, by = "source_window") %>%
    left_join(condition_specs, by = "condition") %>%
    mutate(seed = as.integer(seed), rng_seed = RNG_SEED_BASE + seed, supplied_start_h = case_when(!window_enabled ~
        NA_real_, condition_kind == "permuted" ~ unname(permuted_start_by_window[source_window]), TRUE ~
        (actual_start_h + fixed_shift_h)%%24), supplied_width_h = if_else(window_enabled, actual_width_h,
        NA_real_), signed_window_shift_h = if_else(window_enabled, signed_circular_shift_h(supplied_start_h,
        actual_start_h), NA_real_), absolute_window_shift_h = abs(signed_window_shift_h), method = "modified_pre_fix_peak_trough",
        method_label = "Modified CHIRAL (reference init + window)", fit_source = if_else(condition ==
            "correct_window", "primary_benchmark", "point5_new_fit")) %>%
    arrange(dataset, source_window, condition_order, seed)

stopifnot(nrow(point5_plan) == length(benchmark_datasets) * nrow(source_windows) * nrow(condition_specs) *
    N_SEEDS, all(point5_plan %>%
    count(dataset, source_window, condition) %>%
    pull(n) == N_SEEDS), all(point5_plan$actual_width_h == 12), all(is.na(point5_plan$supplied_start_h[!point5_plan$window_enabled])),
    all(is.finite(point5_plan$supplied_start_h[point5_plan$window_enabled])))


if (isTRUE(RUN_ANALYSIS)) {
    point5_signature <- digest::digest(list(point5_plan, RNG_SEED_BASE, ITERATIONS, benchmark_datasets,
        lapply(as.list(legacy_chiral), function(f) if (is.function(f)) deparse(f) else f)), algo = "sha256")
    log_message <- function(...) message(sprintf(...))
    RUN_CACHE_DIR <- file.path(POINT5_OUTPUT_DIR, "run_cache", point5_signature)
    dir.create(RUN_CACHE_DIR, recursive = TRUE, showWarnings = FALSE)
    run_point5_one <- function(ds, plan_row, iterations = ITERATIONS, n_cores = N_CORES, fit_source = "point5_new_fit") {
        sample_info <- ds$sample_info %>%
            filter(in_circular_window(timepoint_h, plan_row$actual_start_h, plan_row$actual_width_h)) %>%
            filter(sample_id %in% colnames(ds$expr))

        if (nrow(sample_info) < 4L) {
            stop(sprintf("Point 5 plan unexpectedly retained fewer than four samples: %s | %s", ds$id,
                plan_row$source_window))
        }

        E <- ds$expr[, sample_info$sample_id, drop = FALSE]
        E <- E[rowSums(is.na(E)) < ncol(E), , drop = FALSE]

        set.seed(plan_row$rng_seed)
        args <- list(E = E, iterations = iterations, clockgenes = ds$clock_genes, pbar = FALSE, n_cores = n_cores,
            reference_genes = ds$ref_genes, reference_phases = ds$ref_phases, reference_strength = 1,
            use_trough_constraint = TRUE, auto_adjust_anti_phase = TRUE)

        if (isTRUE(plan_row$window_enabled)) {
            args$constrain_12h_window <- TRUE
            args$window_start <- hours_to_rad(plan_row$supplied_start_h)
            args$window_width <- hours_to_rad(plan_row$supplied_width_h)
        }

        fit_warnings <- character()
        result <- tryCatch(withCallingHandlers({
            fit <- NULL
            invisible(capture.output(fit <- suppressMessages(do.call(pre_fix_chiral$CHIRAL_dual_penalty,
                args)), type = "output"))
            fit
        }, warning = function(w) {
            fit_warnings <<- c(fit_warnings, conditionMessage(w))
            invokeRestart("muffleWarning")
        }), error = function(e) e)
        fit_warning_text <- if (length(fit_warnings) == 0L)
            NA_character_ else paste(unique(fit_warnings), collapse = ";")

        common <- tibble(dataset = ds$id, source_window = plan_row$source_window, source_window_label = plan_row$source_window_label,
            condition = plan_row$condition, condition_order = plan_row$condition_order, condition_kind = plan_row$condition_kind,
            condition_label = plan_row$condition_label, seed = plan_row$seed, rng_seed = plan_row$rng_seed,
            method = plan_row$method, method_label = plan_row$method_label, actual_start_h = plan_row$actual_start_h,
            actual_width_h = plan_row$actual_width_h, window_enabled = plan_row$window_enabled, supplied_start_h = plan_row$supplied_start_h,
            supplied_width_h = plan_row$supplied_width_h, signed_window_shift_h = plan_row$signed_window_shift_h,
            absolute_window_shift_h = plan_row$absolute_window_shift_h, fit_source = fit_source, sample_id = sample_info$sample_id,
            group = sample_info$group, truth_h = sample_info$timepoint_h)

        if (inherits(result, "error")) {
            return(common %>%
                mutate(predicted_h = NA_real_, status = "error", error_message = conditionMessage(result),
                  fit_warnings = fit_warning_text, iter = NA_integer_, auto_adjusted = NA))
        }

        common %>%
            mutate(predicted_h = rad_to_hours(result$phi), status = "ok", error_message = NA_character_,
                fit_warnings = fit_warning_text, iter = as.integer(result$iter %||% result$iteration %||%
                  NA_integer_), auto_adjusted = as.logical(result$auto_adjusted %||% NA))
    }

    cache_file_for_plan_row <- function(plan_row) {
        file.path(RUN_CACHE_DIR, sprintf("%s__%s__%s__seed_%02d.rds", plan_row$dataset, plan_row$source_window,
            plan_row$condition, plan_row$seed))
    }

    new_fit_plan <- point5_plan %>%
        filter(condition != "correct_window")

    run_plan_index <- function(i) {
        plan_row <- new_fit_plan[i, ]
        cache_file <- cache_file_for_plan_row(plan_row)
        use_cache <- RESUME_MATCHING_RUNS && !FORCE_RERUN && file.exists(cache_file)
        if (i == 1L || i%%100L == 0L || i == nrow(new_fit_plan)) {
            log_message("Point 5 fit %d/%d: %s | %s | %s | seed %d%s", i, nrow(new_fit_plan), plan_row$dataset,
                plan_row$source_window, plan_row$condition, plan_row$seed, if (use_cache)
                  " [cached]" else "")
        }
        if (use_cache)
            return(readRDS(cache_file))

        result <- run_point5_one(ds = benchmark_datasets[[plan_row$dataset]], plan_row = plan_row, iterations = ITERATIONS,
            n_cores = N_CORES, fit_source = "point5_new_fit")
        saveRDS(result, cache_file)
        result
    }

    log_message("Starting Point 5 stress test: %d new fits, %d seeds, %d outer workers, signature %s",
        nrow(new_fit_plan), N_SEEDS, N_OUTER_WORKERS, point5_signature)

    plan_indices <- seq_len(nrow(new_fit_plan))
    if (.Platform$OS.type != "windows" && N_OUTER_WORKERS > 1L) {
        new_prediction_rows <- parallel::mclapply(plan_indices, run_plan_index, mc.cores = N_OUTER_WORKERS,
            mc.preschedule = FALSE)
    } else {
        new_prediction_rows <- lapply(plan_indices, run_plan_index)
    }
    new_predictions <- bind_rows(new_prediction_rows)

    primary_prediction_path <- file.path(OUTPUT_DIR, "chiral_1_6_3_predictions.csv")
    stopifnot(file.exists(primary_prediction_path))
    primary_predictions <- read_csv(primary_prediction_path, show_col_types = FALSE) %>%
        mutate(error_message = as.character(error_message), auto_adjusted = as.logical(auto_adjusted))

    correct_plan <- point5_plan %>%
        filter(condition == "correct_window")
    correct_predictions <- primary_predictions %>%
        filter(method == "modified_pre_fix_peak_trough", window %in% source_windows$source_window, seed %in%
            POINT5_SEEDS) %>%
        rename(source_window = window) %>%
        select(dataset, source_window, seed, rng_seed, method, sample_id, group, truth_h, predicted_h,
            status, error_message, iter, auto_adjusted) %>%
        inner_join(correct_plan %>%
            select(dataset, source_window, seed, source_window_label, condition, condition_order, condition_kind,
                condition_label, method_label, actual_start_h, actual_width_h, window_enabled, supplied_start_h,
                supplied_width_h, signed_window_shift_h, absolute_window_shift_h, fit_source), by = c("dataset",
            "source_window", "seed")) %>%
        mutate(fit_warnings = NA_character_) %>%
        select(all_of(colnames(new_predictions)))

    stopifnot(nrow(correct_predictions %>%
        distinct(dataset, source_window, condition, seed)) == nrow(correct_plan), all(correct_predictions$status ==
        "ok"))

    # Replay one prespecified correct-window fit through this wrapper and require it to reproduce
    # the existing primary benchmark. This validates the shared data loader, sample subset, seed
    # convention, and production call before the new misspecification conditions are interpreted.
    validation_plan_row <- correct_plan %>%
        filter(dataset == "mouse_tumor", source_window == "12_24h", seed == 1L)
    stopifnot(nrow(validation_plan_row) == 1L)
    validation_replay <- run_point5_one(ds = benchmark_datasets[["mouse_tumor"]], plan_row = validation_plan_row,
        iterations = ITERATIONS, n_cores = N_CORES, fit_source = "point5_validation_replay")
    validation_comparison <- correct_predictions %>%
        filter(dataset == "mouse_tumor", source_window == "12_24h", seed == 1L) %>%
        select(sample_id, primary_predicted_h = predicted_h) %>%
        inner_join(validation_replay %>%
            select(sample_id, replay_predicted_h = predicted_h), by = "sample_id") %>%
        mutate(abs_circular_difference_h = circular_distance_h(primary_predicted_h, replay_predicted_h))
    stopifnot(nrow(validation_comparison) == nrow(validation_replay), max(validation_comparison$abs_circular_difference_h,
        na.rm = TRUE) <= 1e-08)
    write_csv(validation_comparison, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_validation_replay.csv"))

    point5_predictions <- bind_rows(correct_predictions, new_predictions) %>%
        arrange(dataset, source_window, condition_order, seed, sample_id)


} else {
    legacy_point5_predictions <- file.path(LEGACY_POINT5_DIR, "point5_chiral_window_stress_test_predictions.csv")
    stopifnot(file.exists(legacy_point5_predictions))
    point5_predictions <- readr::read_csv(legacy_point5_predictions, show_col_types = FALSE)
    validation_comparison <- tibble::tibble(abs_circular_difference_h = NA_real_)
}
planned_keys <- point5_plan %>%
    distinct(dataset, source_window, condition, seed)
observed_keys <- point5_predictions %>%
    distinct(dataset, source_window, condition, seed)
stopifnot(nrow(anti_join(planned_keys, observed_keys, by = c("dataset", "source_window", "condition",
    "seed"))) == 0L, nrow(anti_join(observed_keys, planned_keys, by = c("dataset", "source_window", "condition",
    "seed"))) == 0L, max(point5_predictions %>%
    count(dataset, source_window, condition, seed, sample_id) %>%
    pull(n)) == 1L)

# Every condition must use the identical actual-window sample subset. Only the supplied prior is
# allowed to change.
sample_set_audit <- point5_predictions %>%
    group_by(dataset, source_window, seed, condition) %>%
    summarise(n_samples = n_distinct(sample_id), sample_set = paste(sort(unique(sample_id)), collapse = "|"),
        truth_set = paste(sprintf("%s=%.12f", sample_id[order(sample_id)], truth_h[order(sample_id)]),
            collapse = "|"), .groups = "drop") %>%
    group_by(dataset, source_window, seed) %>%
    summarise(n_conditions = n(), n_distinct_sample_sets = n_distinct(sample_set), n_distinct_truth_sets = n_distinct(truth_set),
        min_n_samples = min(n_samples), max_n_samples = max(n_samples), .groups = "drop")
stopifnot(all(sample_set_audit$n_conditions == nrow(condition_specs)), all(sample_set_audit$n_distinct_sample_sets ==
    1L), all(sample_set_audit$n_distinct_truth_sets == 1L), all(sample_set_audit$min_n_samples == sample_set_audit$max_n_samples))
write_csv(sample_set_audit, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_sample_set_audit.csv"))
write_csv(point5_predictions, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_predictions.csv"))

point5_run_diagnostics <- point5_predictions %>%
    group_by(dataset, source_window, condition, condition_order, condition_kind, condition_label, seed,
        rng_seed, fit_source) %>%
    summarise(run_status = if_else(all(status == "ok"), "ok", paste(sort(unique(status)), collapse = ";")),
        error_messages = paste(sort(unique(error_message[!is.na(error_message)])), collapse = ";"), warning_messages = paste(sort(unique(fit_warnings[!is.na(fit_warnings)])),
            collapse = ";"), n_samples = n_distinct(sample_id), n_finite_predictions = sum(is.finite(predicted_h)),
        median_iter = median(iter, na.rm = TRUE), reached_iteration_limit = any(iter >= ITERATIONS, na.rm = TRUE),
        n_unique_phase_0_001h = n_distinct(round(predicted_h[is.finite(predicted_h)], 3)), catastrophic_phase_collapse = n_finite_predictions >=
            4L && n_unique_phase_0_001h < 3L, any_auto_adjustment = any(auto_adjusted %in% TRUE, na.rm = TRUE),
        .groups = "drop")
write_csv(point5_run_diagnostics, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_run_diagnostics.csv"))

scorable_predictions <- point5_predictions %>%
    filter(status == "ok", is.finite(predicted_h), is.finite(truth_h))

point5_scored_predictions <- scorable_predictions %>%
    group_by(dataset, source_window, condition, seed) %>%
    group_modify(~score_prediction_group(.x)) %>%
    ungroup()

in_window_h <- function(time_h, start_h, width_h) {
    ifelse(!is.finite(start_h) | !is.finite(width_h), NA, ((time_h - start_h)%%24) <= width_h + 1e-10)
}

distance_to_window_boundary_h <- function(time_h, start_h, width_h) {
    ifelse(!is.finite(start_h) | !is.finite(width_h), NA_real_, pmin(circular_distance_h(time_h, start_h),
        circular_distance_h(time_h, (start_h + width_h)%%24)))
}

distance_outside_window_h <- function(time_h, start_h, width_h) {
    inside <- in_window_h(time_h, start_h, width_h)
    boundary_distance <- distance_to_window_boundary_h(time_h, start_h, width_h)
    ifelse(is.na(inside), NA_real_, ifelse(inside, 0, boundary_distance))
}

point5_scored_predictions <- point5_scored_predictions %>%
    mutate(truth_in_actual_window = in_window_h(truth_h, actual_start_h, actual_width_h), predicted_in_actual_window = in_window_h(predicted_eval_h,
        actual_start_h, actual_width_h), predicted_in_supplied_window = in_window_h(predicted_eval_h,
        supplied_start_h, supplied_width_h), distance_to_actual_boundary_h = distance_to_window_boundary_h(predicted_eval_h,
        actual_start_h, actual_width_h), distance_to_supplied_boundary_h = distance_to_window_boundary_h(predicted_eval_h,
        supplied_start_h, supplied_width_h), distance_outside_actual_window_h = distance_outside_window_h(predicted_eval_h,
        actual_start_h, actual_width_h), distance_outside_supplied_window_h = distance_outside_window_h(predicted_eval_h,
        supplied_start_h, supplied_width_h))

stopifnot(all(point5_scored_predictions$truth_in_actual_window %in% TRUE))
write_csv(point5_scored_predictions, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_scored_predictions.csv"))

safe_mean <- function(x) if (all(is.na(x))) NA_real_ else mean(x, na.rm = TRUE)
safe_median <- function(x) if (all(is.na(x))) NA_real_ else median(x, na.rm = TRUE)
safe_max <- function(x) if (all(is.na(x))) NA_real_ else max(x, na.rm = TRUE)

point5_run_summary <- point5_scored_predictions %>%
    group_by(dataset, source_window, source_window_label, condition, condition_order, condition_kind,
        condition_label, seed, rng_seed, alignment, window_enabled, actual_start_h, actual_width_h, supplied_start_h,
        supplied_width_h, signed_window_shift_h, absolute_window_shift_h) %>%
    summarise(n_samples = n_distinct(sample_id), run_mean_abs_error_h = mean(abs_error_h), run_median_abs_error_h = median(abs_error_h),
        run_rmse_h = sqrt(mean(signed_error_h^2)), run_bias_h = mean(signed_error_h), run_within_2h = mean(abs_error_h <=
            2), run_within_4h = mean(abs_error_h <= 4), run_severe_12h_flip_rate = mean(abs_error_h >=
            10), actual_window_compliance = mean(predicted_in_actual_window), supplied_window_compliance = safe_mean(predicted_in_supplied_window),
        fraction_within_0_25h_actual_boundary = mean(distance_to_actual_boundary_h <= 0.25), fraction_within_0_5h_actual_boundary = mean(distance_to_actual_boundary_h <=
            0.5), fraction_within_1h_actual_boundary = mean(distance_to_actual_boundary_h <= 1), fraction_within_0_25h_supplied_boundary = safe_mean(distance_to_supplied_boundary_h <=
            0.25), fraction_within_0_5h_supplied_boundary = safe_mean(distance_to_supplied_boundary_h <=
            0.5), fraction_within_1h_supplied_boundary = safe_mean(distance_to_supplied_boundary_h <=
            1), median_distance_outside_actual_window_h = median(distance_outside_actual_window_h), max_distance_outside_actual_window_h = max(distance_outside_actual_window_h),
        median_distance_outside_supplied_window_h = safe_median(distance_outside_supplied_window_h),
        max_distance_outside_supplied_window_h = safe_max(distance_outside_supplied_window_h), run_median_iter = median(iter,
            na.rm = TRUE), .groups = "drop")
write_csv(point5_run_summary, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_run_summary.csv"))

point5_condition_summary <- point5_run_summary %>%
    group_by(dataset, source_window, source_window_label, condition, condition_order, condition_kind,
        condition_label, alignment, window_enabled, actual_start_h, actual_width_h, supplied_start_h,
        supplied_width_h, signed_window_shift_h, absolute_window_shift_h) %>%
    summarise(n_runs = n_distinct(seed), median_run_median_abs_error_h = median(run_median_abs_error_h),
        q25_run_median_abs_error_h = quantile(run_median_abs_error_h, 0.25, names = FALSE), q75_run_median_abs_error_h = quantile(run_median_abs_error_h,
            0.75, names = FALSE), median_actual_window_compliance = median(actual_window_compliance),
        median_supplied_window_compliance = safe_median(supplied_window_compliance), median_fraction_within_0_25h_supplied_boundary = safe_median(fraction_within_0_25h_supplied_boundary),
        median_fraction_within_0_5h_supplied_boundary = safe_median(fraction_within_0_5h_supplied_boundary),
        median_fraction_within_1h_supplied_boundary = safe_median(fraction_within_1h_supplied_boundary),
        maximum_distance_outside_supplied_window_h = safe_max(max_distance_outside_supplied_window_h),
        median_iteration = median(run_median_iter, na.rm = TRUE), .groups = "drop") %>%
    arrange(dataset, source_window, condition_order, alignment)
write_csv(point5_condition_summary, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_condition_summary.csv"))

correct_run_summary <- point5_run_summary %>%
    filter(condition == "correct_window") %>%
    select(dataset, source_window, seed, alignment, correct_run_median_abs_error_h = run_median_abs_error_h,
        correct_run_mean_abs_error_h = run_mean_abs_error_h, correct_actual_window_compliance = actual_window_compliance,
        correct_fraction_within_0_25h_actual_boundary = fraction_within_0_25h_actual_boundary)

point5_paired_run_comparison <- point5_run_summary %>%
    filter(condition != "correct_window") %>%
    left_join(correct_run_summary, by = c("dataset", "source_window", "seed", "alignment")) %>%
    mutate(median_error_delta_vs_correct_h = run_median_abs_error_h - correct_run_median_abs_error_h,
        mean_error_delta_vs_correct_h = run_mean_abs_error_h - correct_run_mean_abs_error_h, actual_window_compliance_delta_vs_correct = actual_window_compliance -
            correct_actual_window_compliance, actual_boundary_0_25h_delta_vs_correct = fraction_within_0_25h_actual_boundary -
            correct_fraction_within_0_25h_actual_boundary)
stopifnot(all(is.finite(point5_paired_run_comparison$median_error_delta_vs_correct_h)))
write_csv(point5_paired_run_comparison, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_paired_run_comparison.csv"))

summarise_paired_deltas <- function(df) {
    df %>%
        summarise(n_paired_runs = n(), median_error_delta_vs_correct_h = median(median_error_delta_vs_correct_h),
            error_delta_q25_h = quantile(median_error_delta_vs_correct_h, 0.25, names = FALSE), error_delta_q75_h = quantile(median_error_delta_vs_correct_h,
                0.75, names = FALSE), fraction_runs_worse_than_correct = mean(median_error_delta_vs_correct_h >
                0), median_actual_window_compliance_delta_vs_correct = median(actual_window_compliance_delta_vs_correct),
            median_supplied_window_compliance = safe_median(supplied_window_compliance), median_fraction_within_0_25h_supplied_boundary = safe_median(fraction_within_0_25h_supplied_boundary),
            median_max_distance_outside_supplied_window_h = safe_median(max_distance_outside_supplied_window_h),
            .groups = "drop")
}

point5_paired_cell_summary <- point5_paired_run_comparison %>%
    group_by(dataset, source_window, condition, condition_order, condition_kind, condition_label, alignment,
        signed_window_shift_h, absolute_window_shift_h) %>%
    summarise_paired_deltas() %>%
    arrange(dataset, source_window, condition_order, alignment)
write_csv(point5_paired_cell_summary, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_paired_cell_summary.csv"))

point5_paired_dataset_by_seed <- point5_paired_run_comparison %>%
    group_by(dataset, condition, condition_order, condition_kind, condition_label, alignment, seed) %>%
    summarise(mean_cell_error_delta_vs_correct_h = mean(median_error_delta_vs_correct_h), mean_cell_actual_window_compliance_delta_vs_correct = mean(actual_window_compliance_delta_vs_correct),
        mean_cell_supplied_window_compliance = safe_mean(supplied_window_compliance), .groups = "drop")

point5_paired_dataset_summary <- point5_paired_dataset_by_seed %>%
    group_by(dataset, condition, condition_order, condition_kind, condition_label, alignment) %>%
    summarise(n_seeds = n(), median_seed_mean_error_delta_vs_correct_h = median(mean_cell_error_delta_vs_correct_h),
        error_delta_q25_h = quantile(mean_cell_error_delta_vs_correct_h, 0.25, names = FALSE), error_delta_q75_h = quantile(mean_cell_error_delta_vs_correct_h,
            0.75, names = FALSE), fraction_seeds_worse_than_correct = mean(mean_cell_error_delta_vs_correct_h >
            0), median_seed_mean_actual_window_compliance_delta_vs_correct = median(mean_cell_actual_window_compliance_delta_vs_correct),
        median_seed_mean_supplied_window_compliance = safe_median(mean_cell_supplied_window_compliance),
        .groups = "drop") %>%
    arrange(dataset, condition_order, alignment)
write_csv(point5_paired_dataset_summary, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_paired_dataset_summary.csv"))

point5_paired_overall_by_seed <- point5_paired_run_comparison %>%
    group_by(condition, condition_order, condition_kind, condition_label, alignment, seed) %>%
    summarise(mean_cell_error_delta_vs_correct_h = mean(median_error_delta_vs_correct_h), mean_cell_actual_window_compliance_delta_vs_correct = mean(actual_window_compliance_delta_vs_correct),
        mean_cell_supplied_window_compliance = safe_mean(supplied_window_compliance), mean_cell_fraction_within_0_25h_supplied_boundary = safe_mean(fraction_within_0_25h_supplied_boundary),
        mean_cell_max_distance_outside_supplied_window_h = safe_mean(max_distance_outside_supplied_window_h),
        .groups = "drop")

point5_paired_overall_summary <- point5_paired_overall_by_seed %>%
    group_by(condition, condition_order, condition_kind, condition_label, alignment) %>%
    summarise(n_seeds = n(), median_seed_mean_error_delta_vs_correct_h = median(mean_cell_error_delta_vs_correct_h),
        error_delta_q25_h = quantile(mean_cell_error_delta_vs_correct_h, 0.25, names = FALSE), error_delta_q75_h = quantile(mean_cell_error_delta_vs_correct_h,
            0.75, names = FALSE), fraction_seeds_worse_than_correct = mean(mean_cell_error_delta_vs_correct_h >
            0), median_seed_mean_actual_window_compliance_delta_vs_correct = median(mean_cell_actual_window_compliance_delta_vs_correct),
        median_seed_mean_supplied_window_compliance = safe_median(mean_cell_supplied_window_compliance),
        median_seed_mean_fraction_within_0_25h_supplied_boundary = safe_median(mean_cell_fraction_within_0_25h_supplied_boundary),
        median_seed_mean_max_distance_outside_supplied_window_h = safe_median(mean_cell_max_distance_outside_supplied_window_h),
        .groups = "drop") %>%
    arrange(condition_order, alignment)
write_csv(point5_paired_overall_summary, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_paired_overall_summary.csv"))

point5_condition_overall_by_seed <- point5_run_summary %>%
    group_by(condition, condition_order, condition_kind, condition_label, alignment, seed) %>%
    summarise(mean_cell_median_abs_error_h = mean(run_median_abs_error_h), mean_cell_actual_window_compliance = mean(actual_window_compliance),
        mean_cell_supplied_window_compliance = safe_mean(supplied_window_compliance), mean_cell_fraction_within_0_25h_supplied_boundary = safe_mean(fraction_within_0_25h_supplied_boundary),
        .groups = "drop")

point5_condition_overall_summary <- point5_condition_overall_by_seed %>%
    group_by(condition, condition_order, condition_kind, condition_label, alignment) %>%
    summarise(n_seeds = n(), median_seed_mean_cell_median_abs_error_h = median(mean_cell_median_abs_error_h),
        error_q25_h = quantile(mean_cell_median_abs_error_h, 0.25, names = FALSE), error_q75_h = quantile(mean_cell_median_abs_error_h,
            0.75, names = FALSE), median_seed_mean_actual_window_compliance = median(mean_cell_actual_window_compliance),
        median_seed_mean_supplied_window_compliance = safe_median(mean_cell_supplied_window_compliance),
        median_seed_mean_fraction_within_0_25h_supplied_boundary = safe_median(mean_cell_fraction_within_0_25h_supplied_boundary),
        .groups = "drop") %>%
    arrange(condition_order, alignment)
write_csv(point5_condition_overall_summary, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_condition_overall_summary.csv"))

diagnostic_status_summary <- point5_run_diagnostics %>%
    group_by(condition, condition_order, condition_label) %>%
    summarise(n_runs = n(), n_failed = sum(run_status != "ok"), n_reaching_iteration_limit = sum(reached_iteration_limit),
        n_catastrophic_phase_collapses = sum(catastrophic_phase_collapse), n_with_auto_adjustment = sum(any_auto_adjustment),
        median_iterations = median(median_iter, na.rm = TRUE), .groups = "drop") %>%
    arrange(condition_order)
write_csv(diagnostic_status_summary, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_diagnostic_status_summary.csv"))

stopifnot(all(point5_run_diagnostics$run_status == "ok"), !any(point5_run_diagnostics$catastrophic_phase_collapse),
    nrow(point5_run_summary) == nrow(point5_plan) * 2L, all(point5_condition_summary$n_runs == N_SEEDS))

raw_decision_table <- point5_condition_overall_summary %>%
    filter(alignment == "raw") %>%
    left_join(point5_paired_overall_summary %>%
        filter(alignment == "raw") %>%
        select(condition, median_seed_mean_error_delta_vs_correct_h, error_delta_q25_h, error_delta_q75_h,
            fraction_seeds_worse_than_correct), by = "condition") %>%
    arrange(condition_order)
write_csv(raw_decision_table, file.path(POINT5_OUTPUT_DIR, "point5_chiral_window_stress_test_raw_decision_table.csv"))

log_message("Point 5 stress test completed with %d prediction rows", nrow(point5_predictions))
log_message("Validation replay maximum difference: %.12g h", max(validation_comparison$abs_circular_difference_h))
log_message("Results written to %s", POINT5_OUTPUT_DIR)

print(raw_decision_table %>%
    select(condition, median_seed_mean_cell_median_abs_error_h, median_seed_mean_error_delta_vs_correct_h,
        error_delta_q25_h, error_delta_q75_h, fraction_seeds_worse_than_correct, median_seed_mean_actual_window_compliance,
        median_seed_mean_supplied_window_compliance, median_seed_mean_fraction_within_0_25h_supplied_boundary))
print(diagnostic_status_summary)



## 3. Figure 1D–J

D–G: paired sample-level median absolute circular error across 30 seeds in the 12–24 h
window. H–I: identical six-gene expression against observed time or seed-1 raw predicted
phase in the full-day mouse-tumor cohort. J: median run-level MedAE over 30 seeds across
three cohorts, five windows, and both estimands. These two aggregation orders need not
give identical D–G versus J numbers.

Layout: D–G occupy the first row. H and I are each 52.5 × 69.825 mm; J is 105 × 69.825 mm. Their height is 33% greater than one module, an explicit exception to square-module heights. H–J share the second row (width ratio 1:1:2). Figure 1 uses 8 pt panel titles, 7 pt facet titles, and black 6 pt annotations/subtitles. Other text roles retain their shared sizes. Panel J numbers have no background; their text switches between white and black to maximize contrast against each heatmap cell. In J, adjusted MedAE denotes CHIRAL-native symmetry-adjusted internal-phase recovery, while raw MedAE denotes absolute orientation.


In [ ]:
# Cell 7: Figure 1 inputs, palette, and figure-specific typography
FONT <- FIGURE1_FONT
theme_set(publication_theme())
suppressPackageStartupMessages({
    library(dplyr)
    library(ggplot2)
    library(patchwork)
    library(readr)
    library(scales)
    library(tidyr)
})

required_figure1_files <- c(file.path(BENCHMARK_DIR, "chiral_1_6_3_scored_predictions.csv"), file.path(BENCHMARK_DIR,
    "chiral_1_6_3_summary.csv"), file.path(INPUT_DIR, "Testing_Tumor_Mouse_sample_info.csv"), file.path(INPUT_DIR,
    "Testing_Tumor_Mouse_raw_counts.csv"))
if (!all(file.exists(required_figure1_files))) {
    stop("Missing Figure 1 input(s): ", paste(required_figure1_files[!file.exists(required_figure1_files)],
        collapse = ", "))
}
dir.create(TABLE_DIR, recursive = TRUE, showWarnings = FALSE)

fig1_predictions <- readr::read_csv(file.path(BENCHMARK_DIR, "chiral_1_6_3_scored_predictions.csv"),
    show_col_types = FALSE)
fig1_summary <- readr::read_csv(file.path(BENCHMARK_DIR, "chiral_1_6_3_summary.csv"), show_col_types = FALSE)
fig1_method_levels <- c("original_default", "modified_pre_fix_peak_trough")
fig1_method_labels <- c(original_default = "Original", modified_pre_fix_peak_trough = "Modified")
fig1_method_colours <- c(Original = "#D46A6A", Modified = "#2A9D8F")
fig1_group_colours <- c(Young = "#2A9D8F", Old = "#D46A6A")
fig1_tag_theme <- theme(plot.tag = element_text(family = FONT_FAMILY, size = FONT$tag, face = "bold",
    colour = "black"), plot.tag.position = c(0.02, 0.98))



In [ ]:
# Cell 8: Figure 1D-G paired sample errors
fig1_pointwise_data <- fig1_predictions %>%
    filter(dataset %in% c("mouse_liver_cr_spread", "mouse_tumor"), window == "12_24h", method %in% fig1_method_levels,
        alignment %in% c("best_global", "raw"), status == "ok") %>%
    group_by(dataset, alignment, method, sample_id, truth_h) %>%
    summarise(sample_median_error_h = median(abs_error_h), n_seeds = n_distinct(seed), .groups = "drop") %>%
    mutate(method_label = factor(unname(fig1_method_labels[method]), levels = unname(fig1_method_labels[fig1_method_levels])))
fig1_pair_counts <- fig1_pointwise_data %>%
    count(dataset, alignment, sample_id, name = "n_methods")
if (nrow(fig1_pointwise_data) != 96L || any(fig1_pointwise_data$n_seeds != 30L) || any(fig1_pair_counts$n_methods !=
    2L) || any(!is.finite(fig1_pointwise_data$sample_median_error_h))) {
    stop("Figure 1D-G failed the 30-seed paired sample completeness check.")
}
fig1_pointwise_summary <- fig1_pointwise_data %>%
    group_by(dataset, alignment, method, method_label) %>%
    summarise(n_samples = n(), panel_median_error_h = median(sample_median_error_h), panel_q25_error_h = quantile(sample_median_error_h,
        0.25, names = FALSE), panel_q75_error_h = quantile(sample_median_error_h, 0.75, names = FALSE),
        .groups = "drop")
write_csv(fig1_pointwise_data %>%
    left_join(fig1_pointwise_summary, by = c("dataset", "alignment", "method", "method_label")) %>%
    mutate(method_label = as.character(method_label)) %>%
    arrange(alignment, dataset, sample_id, method), file.path(TABLE_DIR, "figure1_panels_D_G_source_data.csv"))

make_fig1_pointwise_panel <- function(dataset_id, alignment_id, title_text, metric_text) {
    plot_data <- fig1_pointwise_data %>%
        filter(dataset == dataset_id, alignment == alignment_id)
    summary_plot <- fig1_pointwise_summary %>%
        filter(dataset == dataset_id, alignment == alignment_id) %>%
        mutate(label = sprintf("%.2f", panel_median_error_h))
    n_samples <- unique(summary_plot$n_samples)
    ggplot(plot_data, aes(x = method_label, y = sample_median_error_h, group = sample_id)) + geom_line(colour = "#B7C0C5",
        linewidth = 0.34, alpha = 0.78) + geom_point(aes(fill = method_label), shape = 21, size = 1.75,
        stroke = 0.35, colour = "#2E373B") + geom_crossbar(data = summary_plot, aes(x = method_label,
        y = panel_median_error_h, ymin = panel_median_error_h, ymax = panel_median_error_h, group = method_label),
        inherit.aes = FALSE, width = 0.46, linewidth = 0.62, colour = "#111827") + geom_text(data = summary_plot,
        aes(x = method_label, y = 11.55, label = label), inherit.aes = FALSE, family = FONT_FAMILY, fontface = "bold",
        size = FONT$annotation/ggplot2::.pt, colour = ANNOTATION_COLOUR) + scale_fill_manual(values = fig1_method_colours,
        guide = "none") + scale_y_continuous(limits = c(0, 12), breaks = c(0, 3, 6, 9, 12), expand = expansion(mult = c(0,
        0.02))) + labs(title = title_text, subtitle = paste0(metric_text, "
n=", n_samples, "; 30 seeds"),
        x = NULL, y = "Sample-level MedAE (h)") + publication_theme() + theme(axis.line = element_line(linewidth = 0.28,
        colour = "black"), axis.ticks = element_line(linewidth = 0.28, colour = "black"), axis.title.y = element_text(size = FONT$axis_title,
        margin = margin(r = 1.1)), axis.text.y = element_text(size = FONT$tick, colour = "black"), axis.text.x = element_text(size = FONT$tick,
        face = "bold", colour = "black"), panel.grid.major.y = element_line(colour = "#E4E8EA", linewidth = 0.24),
        plot.title = element_text(size = FONT$title, face = "bold", hjust = 0.5, margin = margin(b = 0.3)),
        plot.subtitle = element_text(size = FONT$subtitle, colour = ANNOTATION_COLOUR, hjust = 0.5, margin = margin(b = 1.2)),
        plot.margin = margin(t = 2.1, r = 2, b = 1.2, l = 2))
}

fig1_panel_d <- make_fig1_pointwise_panel("mouse_liver_cr_spread", "best_global", "Mouse liver
CR-spread",
    "Adjusted MedAE (primary)")
fig1_panel_e <- make_fig1_pointwise_panel("mouse_tumor", "best_global", "Mouse tumor", "Adjusted MedAE (primary)")
fig1_panel_f <- make_fig1_pointwise_panel("mouse_liver_cr_spread", "raw", "Mouse liver
CR-spread", "Raw MedAE (separate)")
fig1_panel_g <- make_fig1_pointwise_panel("mouse_tumor", "raw", "Mouse tumor", "Raw MedAE (separate)")



In [ ]:
# Cell 9: Figure 1H-I six-gene expression
if (!requireNamespace("edgeR", quietly = TRUE)) stop("The edgeR package is required to reproduce Figure 1H-I preprocessing.")
fig1_clock_genes <- c("Per1", "Per2", "Per3", "Arntl", "Tef", "Dbp")
fig1_reference_genes <- c("Per1", "Arntl", "Dbp")
fig1_point8_predictions <- fig1_predictions %>%
    filter(dataset == "mouse_tumor", window == "full_24h", method == "modified_pre_fix_peak_trough",
        seed == 1L, alignment == "raw", status == "ok") %>%
    arrange(sample_id)
if (nrow(fig1_point8_predictions) != 12L || any(fig1_point8_predictions$truth_used_to_adjust_prediction) ||
    any(abs(fig1_point8_predictions$predicted_h - fig1_point8_predictions$predicted_eval_h) >= 1e-12)) {
    stop("Figure 1H-I failed the full-24-h, seed-1 raw-phase identity check.")
}
fig1_sample_info <- read.delim(file.path(INPUT_DIR, "Testing_Tumor_Mouse_sample_info.csv"), check.names = FALSE) %>%
    transmute(sample_id = as.character(sample_id), sample_group = as.character(cell), sample_order = as.integer(sub(".*-",
        "", sample_id)), truth_h_from_metadata = as.numeric(timepoint)%%24)
fig1_raw_counts <- read.delim(file.path(INPUT_DIR, "Testing_Tumor_Mouse_raw_counts.csv"), row.names = 1,
    check.names = FALSE)
fig1_expression_z <- t(scale(t(edgeR::cpm(fig1_raw_counts, normalized.lib.sizes = FALSE, log = TRUE,
    prior.count = 0.25))))
fig1_expression_z <- fig1_expression_z[rowSums(!is.finite(fig1_expression_z)) == 0, , drop = FALSE]
if (length(setdiff(fig1_clock_genes, rownames(fig1_expression_z))) != 0L || length(setdiff(fig1_point8_predictions$sample_id,
    colnames(fig1_expression_z))) != 0L) {
    stop("Figure 1H-I count matrix does not contain the required genes or samples.")
}
fig1_expression_wide <- as.data.frame(t(fig1_expression_z[fig1_clock_genes, fig1_point8_predictions$sample_id,
    drop = FALSE]))
fig1_expression_wide$sample_id <- rownames(fig1_expression_wide)
fig1_point8_data <- fig1_expression_wide %>%
    pivot_longer(cols = all_of(fig1_clock_genes), names_to = "gene", values_to = "expression_z") %>%
    left_join(fig1_point8_predictions %>%
        select(sample_id, group, truth_h, predicted_h, method, method_label, seed, rng_seed, window,
            window_label, iter, auto_adjusted), by = "sample_id") %>%
    left_join(fig1_sample_info, by = "sample_id") %>%
    mutate(gene = factor(gene, levels = fig1_clock_genes), sample_group = factor(sample_group, levels = c("Young",
        "Old")), is_reference_gene = as.character(gene) %in% fig1_reference_genes, preprocessing = "edgeR log2 CPM (prior count 0.25), then row-wise z-scoring",
        panel_role = "Qualitative concordance; not independent validation") %>%
    arrange(gene, sample_group, sample_order)
if (nrow(fig1_point8_data) != 72L || any(!is.finite(fig1_point8_data$expression_z))) stop("Figure 1H-I failed the 72-observation completeness check.")
write_csv(fig1_point8_data %>%
    mutate(gene = as.character(gene)), file.path(TABLE_DIR, "figure1_panels_H_I_source_data.csv"))

make_fig1_expression_panel <- function(phase_column, x_label) {
    plot_data <- fig1_point8_data %>%
        mutate(phase_h = .data[[phase_column]]) %>%
        arrange(gene, sample_group, phase_h, sample_order)
    ggplot(plot_data, aes(x = phase_h, y = expression_z, colour = sample_group, shape = sample_group,
        linetype = sample_group, group = sample_group)) + geom_hline(yintercept = 0, colour = "#D9D9D9",
        linewidth = 0.3) + geom_line(linewidth = 0.35, alpha = 0.82) + geom_point(size = 0.95, stroke = 0.3,
        alpha = 0.94) + facet_wrap(~gene, ncol = 3, scales = "fixed", labeller = as_labeller(function(x) ifelse(x == "Arntl", "Bmal1", x))) + scale_x_continuous(limits = c(0,
        24), breaks = c(0, 24), expand = expansion(mult = c(0.04, 0.06))) + scale_y_continuous(breaks = c(-2, 0, 2), expand = expansion(mult = c(0.06, 0.08))) + scale_colour_manual(values = fig1_group_colours,
        name = "Age group") + scale_shape_manual(values = c(Young = 17, Old = 16), name = "Age group") +
        scale_linetype_manual(values = c(Young = "dashed", Old = "solid"), name = "Age group") + labs(x = x_label,
        y = "Z-scored log2 CPM") + publication_theme() + theme(axis.line = element_line(linewidth = 0.3,
        colour = "black"), axis.ticks = element_line(linewidth = 0.3, colour = "black"), axis.title = element_text(size = FONT$axis_title),
        axis.text = element_text(size = FONT$tick, colour = "black"), panel.border = element_rect(colour = "#4D4D4D",
            fill = NA, linewidth = 0.3), strip.background = element_blank(), strip.text = element_text(size = FONT$strip,
            face = "bold"), panel.spacing = grid::unit(2.5, "mm"), legend.position = "bottom", legend.direction = "horizontal",
        legend.title = element_blank(), legend.margin = margin(0, 0, 0, 0), legend.box.spacing = grid::unit(0, "mm"), legend.text = element_text(size = FONT$legend),
        legend.key.width = grid::unit(4, "mm"), legend.key.height = grid::unit(3, "mm"), plot.margin = margin(t = 1, r = 1, b = 1, l = 1, unit = "mm"))
}
fig1_panel_h <- make_fig1_expression_panel("truth_h", "Collection time (h)")
fig1_panel_i <- make_fig1_expression_panel("predicted_h", "CHIRAL raw phase (h)")



In [ ]:
# Cell 10: Figure 1J heatmap and individual/assembled exports
# Choose black or white against the exact continuous fill palette; text has no backing.
fig1_j_fill_scale <- scale_fill_viridis_c(option = "magma", direction = -1,
    limits = c(0, 12), breaks = c(0, 3, 6, 9, 12), oob = scales::squish, name = "MedAE (h)")
fig1_j_text_colour <- function(values) {
    colours <- fig1_j_fill_scale$palette(scales::rescale(scales::squish(values, c(0, 12)), from = c(0, 12)))
    rgb <- grDevices::col2rgb(colours) / 255
    linear_rgb <- ifelse(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055)^2.4)
    luminance <- as.numeric(c(0.2126, 0.7152, 0.0722) %*% linear_rgb)
    black_contrast <- (luminance + 0.05) / 0.05
    white_contrast <- 1.05 / (luminance + 0.05)
    ifelse(black_contrast >= white_contrast, "black", "white")
}
fig1_dataset_levels <- c("mouse_tumor", "mouse_liver_cr_spread", "tumor_pdo")
fig1_dataset_labels <- c(mouse_tumor = "Mouse tumor", mouse_liver_cr_spread = "Mouse liver", tumor_pdo = "Tumor PDO")
fig1_alignment_levels <- c("best_global", "raw")
fig1_metric_labels <- c(best_global = "Adjusted\nMedAE", raw = "Raw\nMedAE")
fig1_window_levels <- c("Full 24h", "0-12h", "6-18h", "12-24h", "18-6h")
fig1_panel_j_data <- fig1_summary %>%
    filter(alignment %in% fig1_alignment_levels, dataset %in% fig1_dataset_levels, method %in% fig1_method_levels) %>%
    transmute(dataset, dataset_label = unname(fig1_dataset_labels[dataset]), window, window_label, method,
        method_label = unname(fig1_method_labels[method]), alignment, metric_label = unname(fig1_metric_labels[alignment]),
        n_samples, n_runs, median_abs_error_h, median_abs_error_q25_h, median_abs_error_q75_h, median_abs_error_min_h,
        median_abs_error_max_h, displayed_value = sprintf("%.2f", median_abs_error_h))
fig1_expected_j <- expand_grid(dataset = fig1_dataset_levels, window_label = fig1_window_levels, method = fig1_method_levels,
    alignment = fig1_alignment_levels)
if (nrow(fig1_panel_j_data) != 60L || nrow(anti_join(fig1_expected_j, fig1_panel_j_data, by = c("dataset",
    "window_label", "method", "alignment"))) > 0L) {
    stop("Figure 1J failed the expected 60-cell completeness check.")
}
fig1_panel_j_data <- fig1_panel_j_data %>%
    mutate(dataset_label = factor(dataset_label, levels = unname(fig1_dataset_labels[fig1_dataset_levels])),
        window_label = factor(window_label, levels = rev(fig1_window_levels)), method_label = factor(method_label,
            levels = unname(fig1_method_labels[fig1_method_levels])), metric_label = factor(metric_label,
            levels = unname(fig1_metric_labels[fig1_alignment_levels])), value_colour = fig1_j_text_colour(median_abs_error_h))
write_csv(fig1_panel_j_data %>%
    mutate(across(where(is.factor), as.character)), file.path(TABLE_DIR, "figure1_panel_J_source_data.csv"))
write_csv(tibble::tribble(~panels, ~role, ~source_description, "D-G", "Paired point-to-point resolution under two estimands",
    "Per-sample MedAE across 30 paired seeds", "H-I", "Qualitative six-gene concordance", "Full-24-h modified CHIRAL seed-1 raw phase",
    "J", "Two-estimand benchmark heatmap", "Median run-level MedAE across 30 paired seeds"), file.path(TABLE_DIR,
    "figure1_notebook_panel_manifest.csv"))
fig1_panel_j <- ggplot(fig1_panel_j_data, aes(x = method_label, y = window_label, fill = median_abs_error_h)) +
    geom_tile(colour = "white", linewidth = 0.28) + geom_text(aes(label = displayed_value, colour = value_colour),
    family = FONT_FAMILY, size = FONT$annotation/ggplot2::.pt, show.legend = FALSE) + facet_grid(rows = vars(metric_label),
    cols = vars(dataset_label)) + scale_colour_identity() + fig1_j_fill_scale +
    labs(x = NULL, y = NULL) + guides(fill = guide_colourbar(direction = "horizontal", title.position = "left",
    title.hjust = 0.5, barwidth = grid::unit(25, "mm"), barheight = grid::unit(2, "mm"), ticks = TRUE)) +
    publication_theme() + theme(panel.grid = element_blank(), panel.spacing = grid::unit(1, "mm"), axis.text.x = element_text(size = FONT$tick,
    colour = "black", angle = 30, hjust = 0.8, vjust = 1), axis.text.y = element_text(size = FONT$tick,
    colour = "black", margin = margin(r = 1)), strip.background = element_rect(fill = "#F2F4F5", colour = NA),
    strip.text.x = element_text(size = FONT$strip, face = "bold", margin = margin(t = 0.8, b = 0.8)),
    strip.text.y = element_text(size = FONT$strip, face = "bold", angle = 0, lineheight = 0.9), legend.position = "bottom", legend.margin = margin(0, 0, 0, 0), legend.box.spacing = grid::unit(0, "mm"),
    legend.title = element_text(size = FONT$legend), legend.text = element_text(size = FONT$legend),
    plot.margin = margin(t = 2.2, r = 1.5, b = 1.2, l = 1.5))

fig1_panels <- list(D = fig1_panel_d, E = fig1_panel_e, F = fig1_panel_f, G = fig1_panel_g, H = fig1_panel_h,
    I = fig1_panel_i, J = fig1_panel_j)
fig1_tagged_panels <- lapply(names(fig1_panels), function(id) fig1_panels[[id]] + labs(tag = id) + fig1_tag_theme)
names(fig1_tagged_panels) <- names(fig1_panels)
# Wrap panels separately to preserve physical 1:1:2 module widths, independent of axes/legends.
assemble_fig1 <- function(panels) {
    top <- wrap_plots(lapply(panels[c("D", "E", "F", "G")], function(p) wrap_elements(full = p)),
        nrow = 1, widths = c(1, 1, 1, 1))
    bottom <- wrap_plots(lapply(panels[c("H", "I", "J")], function(p) wrap_elements(full = p)),
        nrow = 1, widths = c(1, 1, 2))
    (wrap_elements(full = top) / wrap_elements(full = bottom)) + plot_layout(heights = c(1, FIGURE1_BOTTOM_HEIGHT_UNITS))
}
fig1_assembled <- assemble_fig1(fig1_panels)
fig1_assembled_pdf <- assemble_fig1(fig1_tagged_panels)

fig1_exports <- list(D = save_publication(fig1_panels$D, "figure1/Figure1D_mouse_liver_adjusted_error",
    1, 1, pdf_plot = fig1_tagged_panels$D), E = save_publication(fig1_panels$E, "figure1/Figure1E_mouse_tumor_adjusted_error",
    1, 1, pdf_plot = fig1_tagged_panels$E), F = save_publication(fig1_panels$F, "figure1/Figure1F_mouse_liver_raw_error",
    1, 1, pdf_plot = fig1_tagged_panels$F), G = save_publication(fig1_panels$G, "figure1/Figure1G_mouse_tumor_raw_error",
    1, 1, pdf_plot = fig1_tagged_panels$G), H = save_publication(fig1_panels$H, "figure1/Figure1H_known_collection_time",
    1, FIGURE1_BOTTOM_HEIGHT_UNITS, pdf_plot = fig1_tagged_panels$H), I = save_publication(fig1_panels$I, "figure1/Figure1I_modified_chiral_raw_phase",
    1, FIGURE1_BOTTOM_HEIGHT_UNITS, pdf_plot = fig1_tagged_panels$I), J = save_publication(fig1_panels$J, "figure1/Figure1J_two_estimand_heatmap",
    2, FIGURE1_BOTTOM_HEIGHT_UNITS, pdf_plot = fig1_tagged_panels$J), assembled = save_publication(fig1_assembled, "figure1/Figure1_assembled_chiral_benchmark",
    4, 1 + FIGURE1_BOTTOM_HEIGHT_UNITS, pdf_plot = fig1_assembled_pdf))
fig1_source_data <- list(D_G = fig1_pointwise_data, H_I = fig1_point8_data, J = fig1_panel_j_data)


# Restore the shared typography before supplementary figures.
FONT <- DEFAULT_FONT
theme_set(publication_theme())


## 4. Supplementary Figures 1–3

S1 and S2 retain four sampling-window columns and four method/alignment rows. Each is presented as one whole figure, without panel letters, an overall figure title, or an overall subtitle in either PNG or PDF.
Unlabeled component exports retain A–P in filenames only for file identification and
are not displayed individually in the notebook. Truth is a circle and prediction a square; each
sample occupies the same ring across variants within a window. Arrows follow the shortest
circular shift, and adjusted rows use only the benchmark's global transformation.

S3A reports all five cohorts; B reports paired-seed improvements; C separates internal
ordering from absolute-orientation stability; D measures error changes under incorrect
sampling windows; E compares compliance with actual and supplied windows.


In [ ]:
# Cell 11: Supplementary rotation data and plotting helpers
suppressPackageStartupMessages({
    library(dplyr)
    library(tidyr)
    library(readr)
    library(ggplot2)
    library(patchwork)
    library(ragg)
})
METHOD_COLOURS <- c(Original = "#D46A6A", Modified = "#2A9D8F")
benchmark_file <- file.path(BENCHMARK_DIR, "chiral_1_6_3_scored_predictions.csv")
stopifnot(file.exists(benchmark_file))
rotation_raw <- read_csv(benchmark_file, show_col_types = FALSE) %>%
    filter(dataset %in% c("mouse_tumor", "mouse_liver_cr_spread"), window %in% c("0_12h", "6_18h", "12_24h",
        "18_6h"), method %in% c("modified_pre_fix_peak_trough", "original_default"), alignment %in% c("raw",
        "best_global"), seed == 1L, status == "ok")
stopifnot(nrow(rotation_raw) == 336L, nrow(distinct(rotation_raw, dataset, window, method, alignment,
    sample_id)) == 336L)
rotation_raw <- rotation_raw %>%
    mutate(variant = case_when(method == "modified_pre_fix_peak_trough" & alignment == "raw" ~ "Modified raw",
        method == "modified_pre_fix_peak_trough" ~ "Modified adjusted", method == "original_default" &
            alignment == "raw" ~ "Original raw", TRUE ~ "Original adjusted"), method_display = if_else(method ==
        "original_default", "Original", "Modified"), truth_h = truth_h%%24, predicted_h = predicted_eval_h%%24)
ring_map <- rotation_raw %>%
    distinct(dataset, window, sample_id, group, truth_h) %>%
    arrange(dataset, factor(window, levels = c("0_12h", "6_18h", "12_24h", "18_6h")), group, truth_h,
        sample_id) %>%
    group_by(dataset, window) %>%
    mutate(ring = row_number()) %>%
    ungroup()
rotation_data <- rotation_raw %>%
    left_join(ring_map, by = c("dataset", "window", "sample_id", "group", "truth_h")) %>%
    mutate(shift_h = ((predicted_h - truth_h + 12)%%24) - 12, target_h = truth_h + shift_h)
stopifnot(nrow(rotation_data) == 336L, all(is.finite(rotation_data$truth_h)), all(is.finite(rotation_data$predicted_h)))
rotation_manifest <- rotation_data %>%
    distinct(dataset, window, variant, method_display) %>%
    mutate(row = match(variant, c("Modified raw", "Modified adjusted", "Original raw", "Original adjusted")),
        column = match(window, c("0_12h", "6_18h", "12_24h", "18_6h"))) %>%
    arrange(dataset, row, column)
stopifnot(nrow(rotation_manifest) == 32L)
dir.create(TABLE_DIR, recursive = TRUE, showWarnings = FALSE)
write_csv(rotation_data %>%
    arrange(dataset, variant, window, ring), file.path(TABLE_DIR, "SupplementaryFiguresS1_S2_rotation_seed1_source.csv"))
write_csv(rotation_manifest, file.path(TABLE_DIR, "SupplementaryFiguresS1_S2_rotation_manifest.csv"))


In [ ]:
# Cell 12: Supplementary Figures 1-2 whole figures without panel labels
window_names <- c(`0_12h` = "0-12 h", `6_18h` = "6-18 h", `12_24h` = "12-24 h", `18_6h` = "18-6 h")
variant_names <- c(`Modified raw` = "Modified raw", `Modified adjusted` = "Modified adjusted", `Original raw` = "Original raw",
    `Original adjusted` = "Original adjusted")
group_colours <- c(Young = "#2A9D8F", Old = "#D46A6A", Rep_A = "#4C78A8", Rep_B = "#72B7B2")
segment_rows <- function(d) {
    bind_rows(lapply(seq_len(nrow(d)), function(i) {
        z <- d[i, ]
        common <- tibble(ring = z$ring, group = z$group)
        if (z$target_h < 0)
            bind_rows(mutate(common, x = z$truth_h, xend = 0, arrow_end = FALSE), mutate(common, x = 24,
                xend = z$target_h + 24, arrow_end = TRUE)) else if (z$target_h > 24)
            bind_rows(mutate(common, x = z$truth_h, xend = 24, arrow_end = FALSE), mutate(common, x = 0,
                xend = z$target_h - 24, arrow_end = TRUE)) else mutate(common, x = z$truth_h, xend = z$target_h, arrow_end = TRUE)
    }))
}
rotation_panel <- function(d) {
    seg <- segment_rows(d)
    pts <- bind_rows(transmute(d, ring, group, hour = truth_h, phase = "Truth"), transmute(d, ring, group,
        hour = predicted_h, phase = "Prediction"))
    subtitle <- sprintf("n = %d; MedAE = %.2f h", nrow(d), median(d$abs_error_h))
    ggplot() + geom_hline(yintercept = seq_len(max(d$ring)), colour = "#E4E9EC", linewidth = 0.25) +
        geom_segment(data = filter(seg, !arrow_end), aes(x = x, xend = xend, y = ring, yend = ring, colour = group),
            linewidth = 0.52, lineend = "round") + geom_segment(data = filter(seg, arrow_end), aes(x = x,
        xend = xend, y = ring, yend = ring, colour = group), linewidth = 0.52, lineend = "round", arrow = grid::arrow(length = grid::unit(1,
        "mm"), type = "closed")) + geom_point(data = pts, aes(x = hour, y = ring, shape = phase, fill = phase),
        size = 1.35, stroke = 0.3, colour = "#1F2933") + coord_polar(theta = "x", clip = "off") + scale_x_continuous(limits = c(0,
        24), breaks = seq(0, 18, 6), labels = seq(0, 18, 6), expand = c(0, 0)) + scale_y_continuous(limits = c(0,
        max(d$ring) + 0.9), breaks = NULL, expand = c(0, 0)) + scale_colour_manual(values = group_colours,
        name = "Sample group") + scale_shape_manual(values = c(Truth = 21, Prediction = 22), name = "Phase") +
        scale_fill_manual(values = c(Truth = "#2C5AA0", Prediction = "#F4A261"), name = "Phase") + labs(title = paste(variant_names[[d$variant[[1]]]],
        window_names[[d$window[[1]]]]), subtitle = subtitle) + publication_theme() + theme(axis.line = element_blank(),
        axis.ticks = element_blank(), axis.title = element_blank(), axis.text.y = element_blank(), panel.background = element_blank()) +
        theme(panel.grid.major.x = element_line(colour = "#D5DDE2", linewidth = 0.28), axis.text.x = element_text(size = FONT$tick,
            face = "bold"), plot.title = element_text(size = FONT$strip, hjust = 0.5), plot.subtitle = element_text(size = FONT$annotation,
            hjust = 0.5), legend.title = element_text(size = FONT$legend, face = "bold"), legend.text = element_text(size = FONT$legend),
            plot.tag = element_text(size = FONT$tag, face = "bold"), legend.position = "bottom", legend.box = "horizontal",
            legend.key.height = grid::unit(3, "mm"), legend.key.width = grid::unit(4, "mm"), plot.margin = margin(2,
                2, 2, 2))
}
rotation_legend <- function(dataset_id) {
    groups <- if (dataset_id == "mouse_tumor")
        c("Young", "Old") else c("Rep_A", "Rep_B")
    phase_data <- tibble(x = c(1, 4), label_x = c(1.7, 4.7), phase = c("Truth", "Prediction"), label = c("Truth",
        "Prediction"))
    group_data <- tibble(x = c(9, 14), xend = c(11, 16), label_x = c(11.5, 16.5), y = 0.5, group = groups,
        label = groups)
    ggplot() + geom_point(data = phase_data, aes(x = x, y = 0.5, shape = phase, fill = phase), size = 2.1,
        stroke = 0.45, colour = "#1F2933") + geom_text(data = phase_data, aes(x = label_x, y = 0.5, label = label),
        hjust = 0, family = FONT_FAMILY, size = FONT$legend/ggplot2::.pt) + geom_segment(data = group_data,
        aes(x = x, xend = xend, y = y, yend = y, colour = group), linewidth = 0.8, lineend = "round",
        arrow = grid::arrow(length = grid::unit(0.8, "mm"), type = "closed")) + geom_text(data = group_data,
        aes(x = label_x, y = y, label = label), hjust = 0, family = FONT_FAMILY, size = FONT$legend/ggplot2::.pt) +
        scale_shape_manual(values = c(Truth = 21, Prediction = 22)) + scale_fill_manual(values = c(Truth = "#2C5AA0",
        Prediction = "#F4A261")) + scale_colour_manual(values = group_colours[groups]) + coord_cartesian(xlim = c(0,
        20), ylim = c(0, 1), clip = "off") + publication_theme() + theme(axis.line = element_blank(),
        axis.ticks = element_blank(), axis.text = element_blank(), axis.title = element_blank(), panel.grid = element_blank(),
        legend.position = "none", plot.margin = margin(1, 2, 1, 2, unit = "mm"))
}
make_rotation_figure <- function(dataset_id, figure_number) {
    dset <- filter(rotation_data, dataset == dataset_id)
    plots <- list()
    stems <- character()
    ix <- 0L
    for (variant in variant_names) for (window in names(window_names)) {
        ix <- ix + 1L
        key <- filter(dset, variant == !!variant, window == !!window)
        p <- rotation_panel(key)
        letter <- LETTERS[ix]
        stem <- sprintf("supplementary%d/FigureS%d%s_%s_%s", figure_number, figure_number, letter, tolower(gsub(" ",
            "_", variant)), window)
        panel_png <- p + theme(legend.position = "none")
        # S1/S2 are whole figures: retain unlabeled component files, but display only the whole figure.
        save_publication(panel_png, stem, 1, 1, pdf_plot = panel_png,
            pdf_has_panel_labels = FALSE, display_inline = FALSE)
        plots[[ix]] <- panel_png
        stems[ix] <- stem
    }
    assembled <- (wrap_plots(plots, ncol = 4)/rotation_legend(dataset_id)) + plot_layout(heights = c(4,
        0.25))
    save_publication(assembled, sprintf("supplementary%d/FigureS%d_assembled_%s_rotation", figure_number,
        figure_number, if_else(dataset_id == "mouse_tumor", "mouse_tumor", "mouse_liver")), 4, 5, pdf_plot = assembled, pdf_has_panel_labels = FALSE)
    list(dataset = dataset_id, panels = plots, panel_stems = stems, assembled = assembled, source = dset)
}
supp1 <- make_rotation_figure("mouse_tumor", 1)
supp2 <- make_rotation_figure("mouse_liver_cr_spread", 2)
write_csv(supp1$source %>%
    arrange(variant, window, ring), file.path(TABLE_DIR, "FigureS1_mouse_tumor_rotation_source.csv"))
write_csv(supp2$source %>%
    arrange(variant, window, ring), file.path(TABLE_DIR, "FigureS2_mouse_liver_rotation_source.csv"))


In [ ]:
# Cell 13: Supplementary Figure 3 benchmark and robustness panels
summary_file <- file.path(BENCHMARK_DIR, "chiral_1_6_3_summary.csv")
seed_file <- file.path(BENCHMARK_DIR, "chiral_1_6_3_seed_stability_overall_paired.csv")
concordance_file <- file.path(BENCHMARK_DIR, "chiral_1_6_3_run_concordance_summary.csv")
stress_file <- file.path(POINT5_DIR, "point5_chiral_window_stress_test_raw_decision_table.csv")
stopifnot(all(file.exists(c(summary_file, seed_file, concordance_file, stress_file))))
dataset_labels <- c(mouse_tumor = "Mouse tumor", mouse_liver_cr_spread = "Mouse liver\nCR-spread", tumor_cell = "Human tumor\ncell lines",
    tumor_pdo = "Human tumor PDO", normal_wb_training = "Human normal\nwhole blood")
method_labels <- c(original_default = "Original", modified_pre_fix_peak_trough = "Modified")
windows <- c("Full 24h", "0-12h", "6-18h", "12-24h", "18-6h")
a_data <- read_csv(summary_file, show_col_types = FALSE) %>%
    filter(alignment %in% c("best_global", "raw"), dataset %in% names(dataset_labels), method %in% names(method_labels),
        window_label %in% windows) %>%
    mutate(dataset_label = factor(unname(dataset_labels[dataset]), levels = unname(dataset_labels)),
        method_label = factor(unname(method_labels[method]), levels = c("Modified", "Original")), estimand = factor(if_else(alignment ==
            "best_global", "CHIRAL-native\nadjusted MedAE", "Raw MedAE\nabsolute orientation"), levels = c("CHIRAL-native\nadjusted MedAE",
            "Raw MedAE\nabsolute orientation")), window_label = factor(window_label, levels = windows),
        label = sprintf("%.2f", median_abs_error_h), text_colour = if_else(median_abs_error_h >= 5.5,
            "white", "black"))
stopifnot(nrow(a_data) == 100L, nrow(distinct(a_data, dataset, window_label, method, alignment)) == 100L)
b_data <- read_csv(seed_file, show_col_types = FALSE) %>%
    filter(alignment %in% c("raw", "best_global")) %>%
    transmute(alignment, metric = factor(if_else(alignment == "best_global", "CHIRAL-native\nadjusted MedAE",
        "Raw MedAE\nabsolute orientation")), seed, delta = delta_original_minus_modified_h)
stopifnot(nrow(b_data) == 60L, all(is.finite(b_data$delta)), nrow(distinct(b_data, alignment, seed)) ==
    60L, all(count(b_data, alignment)$n == 30L))
c_data <- read_csv(concordance_file, show_col_types = FALSE) %>%
    filter(method %in% names(method_labels)) %>%
    select(dataset, window, method, n_comparisons_to_seed1, median_raw_run_distance_h, median_symmetry_adjusted_run_distance_h) %>%
    pivot_longer(c(median_raw_run_distance_h, median_symmetry_adjusted_run_distance_h), names_to = "metric",
        values_to = "distance_h") %>%
    mutate(metric = factor(recode(metric, median_raw_run_distance_h = "Absolute orientation\n(raw)",
        median_symmetry_adjusted_run_distance_h = "Internal ordering\n(symmetry-adjusted)"), levels = c("Internal ordering\n(symmetry-adjusted)",
        "Absolute orientation\n(raw)")), method_label = factor(unname(method_labels[method]), levels = c("Original",
        "Modified")))
stopifnot(nrow(c_data) == 100L, all(c_data$n_comparisons_to_seed1 == 29L))
stress_raw <- read_csv(stress_file, show_col_types = FALSE) %>%
    filter(alignment == "raw")
stopifnot(nrow(stress_raw) == 9L, all(stress_raw$n_seeds == 30L))
condition_labels <- c(correct_window = "Correct window", window_off = "Window disabled", shift_minus_4h = "Shift -4 h",
    shift_minus_2h = "Shift -2 h", shift_plus_2h = "Shift +2 h", shift_plus_4h = "Shift +4 h", shift_plus_6h = "Shift +6 h",
    shift_plus_12h = "Shift +12 h", permuted_window_labels = "Permuted labels")
d_data <- stress_raw %>%
    filter(condition != "correct_window") %>%
    mutate(condition_display = factor(unname(condition_labels[condition]), levels = rev(unname(condition_labels[condition[order(condition_order)]]))),
        stress_class = case_when(condition == "window_off" ~ "Window disabled", condition %in% c("shift_minus_2h",
            "shift_plus_2h") ~ "Small shift", condition %in% c("shift_plus_12h", "permuted_window_labels") ~
            "Gross mismatch", TRUE ~ "Other shift"))
e_levels <- rev(c("Window disabled", "Correct window", "Shift -4 h", "Shift -2 h", "Shift +2 h", "Shift +4 h",
    "Shift +6 h", "Shift +12 h", "Permuted labels"))
e_wide <- stress_raw %>%
    mutate(condition_display = factor(unname(condition_labels[condition]), levels = e_levels)) %>%
    transmute(condition, condition_order, condition_display, actual = median_seed_mean_actual_window_compliance,
        supplied = median_seed_mean_supplied_window_compliance)
e_data <- e_wide %>%
    pivot_longer(c(actual, supplied), names_to = "type", values_to = "compliance") %>%
    filter(is.finite(compliance)) %>%
    mutate(type = factor(recode(type, actual = "Actual cohort window", supplied = "Supplied inference window"),
        levels = c("Actual cohort window", "Supplied inference window")))
stopifnot(nrow(d_data) == 8L, nrow(e_data) == 17L)
plot_a <- ggplot(a_data, aes(window_label, method_label, fill = median_abs_error_h)) + geom_tile(colour = "white",
    linewidth = 0.35) + geom_text(aes(label = label, colour = text_colour), size = FONT$annotation/ggplot2::.pt,
    family = FONT_FAMILY, show.legend = FALSE) + facet_grid(dataset_label ~ estimand) + scale_colour_identity() +
    scale_fill_viridis_c(option = "magma", direction = -1, limits = c(0, 12), name = "MedAE (h)") + labs(title = "Internal-phase recovery and absolute orientation\nare separate estimands",
    x = "Sampling window", y = NULL) + publication_theme() + theme(panel.grid = element_blank(), strip.background = element_rect(fill = "#F0F2F3"),
    strip.text.y = element_text(angle = 0, size = FONT$strip, lineheight = 0.9), strip.text.x = element_text(size = FONT$strip,
        lineheight = 0.9), legend.position = "bottom", axis.text.x = element_text(size = FONT$tick),
    axis.text.y = element_text(size = FONT$tick))
plot_b <- ggplot(b_data, aes(metric, delta, fill = metric)) + geom_hline(yintercept = 0, linetype = "dashed",
    linewidth = 0.35) + geom_boxplot(width = 0.52, alpha = 0.25, outlier.shape = NA) + geom_point(aes(colour = metric),
    position = position_jitter(width = 0.08, seed = RNG_SEED_BASE + 301L), size = 1.1, alpha = 0.8) +
    stat_summary(fun = median, geom = "point", shape = 23, size = 2.3, fill = "white") + scale_fill_manual(values = c("#6F7F8C",
    "#264653")) + scale_colour_manual(values = c("#6F7F8C", "#264653")) + labs(title = "Paired-seed improvement",
    subtitle = "Thirty paired seeds; positive values favour modified CHIRAL", x = NULL, y = "Original - modified MedAE (h)") +
    publication_theme() + theme(legend.position = "none")
plot_c <- ggplot(c_data, aes(metric, distance_h, fill = method_label, colour = method_label)) + geom_boxplot(position = position_dodge(0.68),
    width = 0.55, alpha = 0.25, outlier.shape = NA) + geom_point(position = position_jitterdodge(0.09,
    dodge.width = 0.68, seed = RNG_SEED_BASE + 302L), size = 0.8, alpha = 0.6) + scale_y_log10() + scale_fill_manual(values = METHOD_COLOURS) +
    scale_colour_manual(values = METHOD_COLOURS) + labs(title = "Internal-order and orientation stability\nare distinct",
    subtitle = "Cell-level median distance to seed 1 (25 cells)", x = NULL, y = "Run-to-seed-1 distance (h; log scale)",
    fill = NULL, colour = NULL) + publication_theme() + theme(legend.position = "bottom")
stress_cols <- c(`Window disabled` = "#6B7280", `Small shift` = "#2A9D8F", `Other shift` = "#4C78A8",
    `Gross mismatch` = "#C44E52")
plot_d <- ggplot(d_data, aes(y = condition_display)) + geom_vline(xintercept = 0, linetype = "dashed",
    linewidth = 0.35) + geom_segment(aes(x = error_delta_q25_h, xend = error_delta_q75_h, yend = condition_display,
    colour = stress_class), linewidth = 0.7) + geom_point(aes(x = median_seed_mean_error_delta_vs_correct_h,
    colour = stress_class), size = 1.9) + scale_colour_manual(values = stress_cols) + labs(title = "Window errors change\nexact-time accuracy",
    subtitle = "Median and IQR across 30 paired seeds;\npositive values are worse", x = "Raw MedAE change versus\ncorrect window (h)",
    y = NULL, colour = NULL) + publication_theme() + theme(legend.position = "bottom")
plot_e <- ggplot() + geom_segment(data = filter(e_wide, is.finite(actual), is.finite(supplied)), aes(x = actual,
    xend = supplied, y = condition_display, yend = condition_display), linewidth = 0.45, colour = "#D9DEE3") +
    geom_point(data = e_data, aes(compliance, condition_display, colour = type, shape = type), size = 1.9,
        stroke = 0.6) + scale_colour_manual(values = c(`Actual cohort window` = "#264653", `Supplied inference window` = "#D9772B")) +
    scale_shape_manual(values = c(`Actual cohort window` = 16, `Supplied inference window` = 1)) + scale_x_continuous(limits = c(0.3,
    0.7), labels = scales::label_percent(accuracy = 1)) + labs(title = "Supplied-window compliance is not\nindependent validation",
    subtitle = "Wrong windows can agree with the supplied prior", x = "Predictions within window (%)",
    y = NULL, colour = NULL, shape = NULL) + publication_theme() + theme(legend.position = "bottom")
individual <- list(A = list(plot_a, "FigureS3A_two_estimand_benchmark", 4, 2), B = list(plot_b, "FigureS3B_paired_seed_improvement",
    2, 2), C = list(plot_c, "FigureS3C_cross_seed_concordance", 2, 2), D = list(plot_d, "FigureS3D_window_stress_accuracy",
    2, 2), E = list(plot_e, "FigureS3E_window_compliance", 2, 2))
for (letter in names(individual)) {
    z <- individual[[letter]]
    individual_plot <- switch(letter, A = z[[1]], B = z[[1]] + theme(legend.position = "none"), C = z[[1]] +
        theme(legend.position = "bottom"), D = z[[1]] + guides(colour = guide_legend(nrow = 2)) + theme(legend.position = "bottom"),
        E = z[[1]] + guides(colour = guide_legend(nrow = 2), shape = guide_legend(nrow = 2)) + theme(legend.position = "bottom"))
    save_publication(individual_plot, file.path("supplementary3", z[[2]]), z[[3]], z[[4]], individual_plot +
        labs(tag = letter) + theme(plot.tag = element_text(family = FONT_FAMILY, size = FONT$tag, face = "bold")))
}
supp3 <- plot_a + plot_b + plot_c + plot_d + plot_e + plot_layout(design = "AAAA
AAAA
BBCC
BBCC
DDEE
DDEE",
    widths = c(1, 1, 1, 1), heights = c(1, 1, 1, 1, 1, 1))
supp3_tagged <- supp3 + plot_annotation(tag_levels = "A", theme = theme(plot.tag = element_text(family = FONT_FAMILY,
    size = FONT$tag, face = "bold")))
save_publication(supp3, "supplementary3/FigureS3_assembled_reviewer_resolution", 4, 6, supp3_tagged)
write_csv(a_data, file.path(TABLE_DIR, "FigureS3A_two_estimand_benchmark.csv"))
write_csv(b_data, file.path(TABLE_DIR, "FigureS3B_paired_seed_improvement.csv"))
write_csv(c_data, file.path(TABLE_DIR, "FigureS3C_cross_seed_concordance.csv"))
write_csv(d_data, file.path(TABLE_DIR, "FigureS3D_window_stress_accuracy.csv"))
write_csv(e_data, file.path(TABLE_DIR, "FigureS3E_window_compliance.csv"))


## 5. Export checks and session information


In [ ]:
# Cell 14: Export validation, provenance, and session information
export_manifest <- bind_rows(EXPORT_MANIFEST)
stopifnot(nrow(export_manifest) == 48L, !anyDuplicated(export_manifest$stem), all(export_manifest$minimum_text_pt >=
    ifelse(grepl("^figure1/", export_manifest$stem), min(unlist(FIGURE1_FONT)), min(unlist(DEFAULT_FONT))) - 1e-06))
generated_files <- file.path(FIGURE_DIR, c(export_manifest$png, export_manifest$pdf))
stopifnot(all(file.exists(generated_files)), all(file.info(generated_files)$size > 0))
write_csv(export_manifest, file.path(TABLE_DIR, "publication_export_manifest.csv"))
font_configuration <- bind_rows(
    tibble(scope = "default", role = names(DEFAULT_FONT), font_size_pt = unlist(DEFAULT_FONT), family = FONT_FAMILY),
    tibble(scope = "Figure1", role = names(FIGURE1_FONT), font_size_pt = unlist(FIGURE1_FONT), family = FONT_FAMILY))
write_csv(font_configuration, file.path(TABLE_DIR, "publication_font_configuration.csv"))

primary_inputs <- c("Testing_Tumor_Mouse_sample_info.csv", "Testing_Tumor_Mouse_raw_counts.csv", "mouse_liver/cr_spread_6mon.txt",
    "Testing_Tumor_Cell_sample_info.csv", "Testing_Tumor_Cell_raw_counts.csv", "Testing_Tumor_PDO_sample_info.csv",
    "Testing_Tumor_PDO_raw_TPM.txt", "Training_Normal_WB_sample_info.csv", "Training_Normal_WB_raw_TPM.txt")
input_manifest <- tibble(input_path_relative_to_INPUT_DIR = primary_inputs, bytes = file.info(file.path(INPUT_DIR,
    primary_inputs))$size, sha256 = vapply(file.path(INPUT_DIR, primary_inputs), digest::digest, character(1),
    algo = "sha256", file = TRUE), provenance = "Existing processed benchmark matrix/metadata; process.smk linkage not established")
write_csv(input_manifest, file.path(TABLE_DIR, "publication_input_manifest.csv"))
if (!RUN_ANALYSIS) {
    cache_inputs <- c("chiral_1_6_3_predictions.csv", "chiral_1_6_3_scored_predictions.csv", "point5_window_stress_test/point5_chiral_window_stress_test_predictions.csv")
    cache_manifest <- tibble(cache_path_relative_to_LEGACY_BENCHMARK_DIR = cache_inputs, bytes = file.info(file.path(LEGACY_BENCHMARK_DIR,
        cache_inputs))$size, sha256 = vapply(file.path(LEGACY_BENCHMARK_DIR, cache_inputs), digest::digest,
        character(1), algo = "sha256", file = TRUE))
    write_csv(cache_manifest, file.path(TABLE_DIR, "publication_prediction_cache_manifest.csv"))
}
writeLines(capture.output(sessionInfo()), file.path(TABLE_DIR, "publication_session_info.txt"))
print(export_manifest %>%
    count(figure = dirname(stem), name = "panel_and_assembly_pairs"))
cat("Validated", nrow(export_manifest), "PNG/PDF pairs; Figure 1 annotations/subtitles are 6 pt; other text is at least 7 pt.\n")
sessionInfo()

